## Initial data processing for quality and leakage prevention

In [ ]:
import pandas as pd
import json
from pathlib import Path
import re


# Carregar o ficheiro JSON

#Do CSM 
csm_ic = Path('./../../data/raw_data/csm_incumprimento_contratos_20251029_114925.json')
csm_dv = Path('./../../data/raw_data/csm_violencia_domestica_20251029_131345.json')

#Do DGSI
#Violencia Doméstica
dgsi_dv_trp = Path('./../../data/raw_data/dgsi_violencia_domestica_trp_20251013_175933.json')
dgsi_dv_trl = Path('./../../data/raw_data/dgsi_violencia_domestica_trl_20251013_173521.json')
dgsi_dv_trc = Path('./../../data/raw_data/dgsi_violencia_domestica_trc_20251014_104127.json')
dgsi_dv_tre = Path('./../../data/raw_data/dgsi_violencia_domestica_tre_20251013_171439.json')
dgsi_dv_trg = Path('./../../data/raw_data/dgsi_violencia_domestica_trg_20251014_104936.json')
dgsi_dv_stj = Path('./../../data/raw_data/dgsi_violencia_domestica_stj_20251016_093545.json')

#Incumprimento de Contratos
dgsi_ic_jp = Path('./../../data/raw_data/dgsi_incumprimento_contratos_20251028_130853.json')
dgsi_ic_trl = Path('./../../data/raw_data/dgsi_incumprimento_contratos_trl_20251028_152622.json')
dgsi_ic_trp = Path('./../../data/raw_data/dgsi_incumprimento_contratos_trp_20251028_162602.json')
dgsi_ic_trc = Path('./../../data/raw_data/dgsi_incumprimento_contratos_trc_20251028_154117.json')
dgsi_ic_tre = Path('./../../data/raw_data/dgsi_incumprimento_contratos_tre_20251028_141118.json')
dgsi_ic_trg = Path('./../../data/raw_data/dgsi_incumprimento_contratos_trg_20251028_171624.json')
dgsi_ic_stj = Path('./../../data/raw_data/dgsi_incumprimento_contratos_stj_20251028_135712.json')

ic_cases_paths = [
    csm_ic,
    dgsi_ic_jp,
    dgsi_ic_trl,
    dgsi_ic_trp,
    dgsi_ic_trc,
    dgsi_ic_tre,
    dgsi_ic_trg,
    dgsi_ic_stj,
    
    
] 

dv_cases_paths = [
    csm_dv,
    dgsi_dv_trp,
    dgsi_dv_trl,
    dgsi_dv_trc,
    dgsi_dv_tre,
    dgsi_dv_trg,
    dgsi_dv_stj,
    ]


# Verificar se o ficheiro existe
def create_dataframe_from_paths(paths):
    dataframes = []
    for i, path in enumerate(paths):
        if not path.exists():
            print(f'Error: File not found at {path.absolute()}')
            print(f'Current working directory: {Path.cwd()}')
        else:
            with open(path, encoding='utf-8') as f:
                data = json.load(f)
            
            acordaos = pd.json_normalize(data, sep='_')
            dataframes.append(acordaos)
    
    return dataframes




In [ ]:
dataframes_dv = create_dataframe_from_paths(dv_cases_paths) 

In [ ]:
dataframes_ic = create_dataframe_from_paths(ic_cases_paths)

In [ ]:


def concatenate_case_dataframes():
    """
    Concatena todos os dataframes de cada tipo de caso em 2 dataframes finais:
    - df_dv_all: Todos os casos de Violência Doméstica
    - df_ic_all: Todos os casos de Incumprimento de Contratos
    
    Returns:
        tuple: (df_dv_all, df_ic_all)
    """
    # Violência Doméstica - concatenar todos os tribunais
    df_dv_all = pd.concat([
        dataframes_dv[0],  # CSM
        dataframes_dv[1],  # TRP
        dataframes_dv[2],  # TRL
        dataframes_dv[3],  # TRC
        dataframes_dv[4],  # TRE
        dataframes_dv[5],  # TRG
        dataframes_dv[6],  # STJ
    ], ignore_index=True)
    
    print(f" Violência Doméstica: {len(df_dv_all)} casos totais")
    print(f"   - CSM: {len(dataframes_dv[0])} casos")
    print(f"   - TRP: {len(dataframes_dv[1])} casos")
    print(f"   - TRL: {len(dataframes_dv[2])} casos")
    print(f"   - TRC: {len(dataframes_dv[3])} casos")
    print(f"   - TRE: {len(dataframes_dv[4])} casos")
    print(f"   - TRG: {len(dataframes_dv[5])} casos")
    print(f"   - STJ: {len(dataframes_dv[6])} casos\n")
    
    # Incumprimento de Contratos - concatenar todos os tribunais
    df_ic_all = pd.concat([
        dataframes_ic[0],  # CSM
        dataframes_ic[1],  # JP
        dataframes_ic[2],  # TRL
        dataframes_ic[3],  # TRP
        dataframes_ic[4],  # TRC
        dataframes_ic[5],  # TRE
        dataframes_ic[6],  # TRG
        dataframes_ic[7],  # STJ
    ], ignore_index=True)
    
    print(f" Incumprimento de Contratos: {len(df_ic_all)} casos totais")
    print(f"   - CSM: {len(dataframes_ic[0])} casos")
    print(f"   - JP: {len(dataframes_ic[1])} casos")
    print(f"   - TRL: {len(dataframes_ic[2])} casos")
    print(f"   - TRP: {len(dataframes_ic[3])} casos")
    print(f"   - TRC: {len(dataframes_ic[4])} casos")
    print(f"   - TRE: {len(dataframes_ic[5])} casos")
    print(f"   - TRG: {len(dataframes_ic[6])} casos")
    print(f"   - STJ: {len(dataframes_ic[7])} casos\n")
    
    
    print(f"Total cases combined: {len(df_dv_all) + len(df_ic_all)}")
    
    return df_dv_all, df_ic_all


# Criar os 2 dataframes consolidados
df_dv_all, df_ic_all = concatenate_case_dataframes()

# Escolher qual tipo de caso processar
type_case = 'dv'  # 'dv' para Violência Doméstica, 'ic' para Incumprimento de Contratos

if type_case == 'dv':
    df = df_dv_all
    print(f"🔍 Processando: Violência Doméstica ({len(df)} casos)")
elif type_case == 'ic':
    df = df_ic_all
    print(f"🔍 Processando: Incumprimento de Contratos ({len(df)} casos)")
else:
    raise ValueError(f"Tipo de caso inválido: {type_case}. Use 'dv' ou 'ic'.")



## Preparação Inicial dos df's para CLassificação e EDA


In [ ]:
df

In [ ]:
def check_proportion_and_filter(df):
    prop_texto_int_disp = df['texto_integral_disponivel'].isnull().sum() / len(df) # Proporção de valores nulos na coluna 'texto_integral_disponivel'
    print(f'Proporção de valores nulos em texto_integral_disponivel: {prop_texto_int_disp:.2%}')
    
    
    prop_texto_int_disp_falso = (df['texto_integral_disponivel'] == "N").sum() / len(df) # Proporção de valores "N" na coluna 'texto_integral_disponivel'
    print(f'Proporção de valores N em texto_integral_disponivel: {prop_texto_int_disp_falso:.2%}')
    
    # prop_texto_int = df['texto_integral_completo'].isnull().sum() / len(df) # Proporção de valores nulos na coluna 'texto_integral_disponivel'
    # print(f'Proporção de valores nulos em texto_integral_completo: {prop_texto_int:.2%}')
    
    prop_decisao_extraida = df['decisao_extraida_do_texto_integral'].isnull().sum() / len(df) # Proporção de valores nulos na coluna 'decisao_extraida_texto_integral'
    print(f'Proporção de valores nulos em decisao_extraida_do_texto: {prop_decisao_extraida:.2%}')

    
    # Create a new dataframe with cases where full text is available (NOT 'N') but decision not extracted
    mask_text_available = df['texto_integral_disponivel'] != 'N'

    df_text_available_no_decision = df[mask_text_available & df['decisao_extraida_do_texto_integral'].isnull()].copy()

    prop_text_available_no_decision = len(df_text_available_no_decision) / len(df)
    print(f'Proporção de casos com texto integral disponível (não "N") mas sem decisão extraída: {prop_text_available_no_decision:.2%}')
    print(f'Numero de casos com texto integral disponível (não "N") mas sem decisão extraída: {len(df_text_available_no_decision)}')

    return df_text_available_no_decision
    

In [ ]:
# Excluir JP apenas para aplicar o drop; guardar os JP à parte
df_jp = df[df['tribunal'].str.startswith('JP_', na=False)].copy()
df_non_jp = df[~df['tribunal'].str.startswith('JP_', na=False)].copy()  #~df means ignoring the condition , getting all values different than the condition

# Drop rows com texto_integral_disponivel ausente apenas nos não-JP
df_non_jp = df_non_jp.dropna(subset=['texto_integral_disponivel'], how='all')

# Reunir novamente: manter todos os JP intactos + non-JP filtrados
df = pd.concat([df_non_jp, df_jp], axis=0)

# Opcional: restaurar a ordem original dos índices
df = df.sort_index()

df

In [ ]:
# Capturar o dataframe retornado pela função
df_casos_problematicos = check_proportion_and_filter(df)

print(f'Temos {len(df_casos_problematicos)} casos problemáticos onde o texto integral está disponível mas a decisão não foi extraída.')

In [ ]:
df_casos_problematicos

In [ ]:
# Drop rows with missing data in column: 'decisao_extraida_do_texto_integral'
# df_casos_fixed = df_casos_fixed.dropna(subset=['decisao_extraida_do_texto_integral'], how='all')

df_casos_fixed = df.dropna(subset=['decisao_extraida_do_texto_integral'], how='all')

df_casos_fixed

In [ ]:
df_fixed = df.copy()

# Assuming 'id' is the common key column; adjust as needed

df_fixed.set_index('url', inplace=True)
df_casos_fixed.set_index('url', inplace=True)

# Update df with values from df_casos
df_fixed.update(df_casos_fixed)

# Reset index if needed
df_fixed.reset_index(inplace=True)
df_casos_fixed.reset_index(inplace=True)

In [ ]:
# drop rows where the column equals "N"
df_fixed = df_fixed[df_fixed['texto_integral_disponivel'] != 'N']



df_fixed

In [ ]:
# Drop rows with missing data in column: 'decisao_extraida_do_texto_integral'
df_fixed = df_fixed.dropna(subset=['decisao_extraida_do_texto_integral'], how='all')

# Drop duplicate rows based on 'n_processo' column, keeping distinct cases
# URL is not enought to identify distinct cases as some cases have multiple URLs due to scraping being done on different databases
df_fixed = df_fixed.drop_duplicates(subset=['n_processo'])

df_fixed

In [ ]:
df_fixed

In [ ]:
def extract_decision_summary(verbose_decision):
    if not verbose_decision:
        return None
    
    # Convert to lowercase for matching
    text = verbose_decision.lower()
    # Remove punctuation for better matching
    text_clean = re.sub(r'[^\w\s]', ' ', text)
    
    # PRIORITY 1: Check for PARTIAL patterns first
    partial_patterns = [
        r'\bparcial',                           # Partial
        r'\bparcialmente',                      # Partially
        r'\bem\s+parte',                        # In part
        r'\brevogad[oa]\s+parcial',            # Partially revoked
        r'\bprocedente\s+parcial',             # Partially procedent
        r'\bprovido\s+parcial',                # Partially granted
        r'\bconcedid[oa]\s+parcial',           # Partially granted
    ]
    
    # If ANY partial pattern found, return PARCIAL immediately
    for pattern in partial_patterns:
        if re.search(pattern, text_clean):
            return 'PARCIALMENTE PROCEDENTE'
    
    # PRIORITY 2: Check for FAVORABLE patterns (complete victory - explicitly exclude partials)
    favorable_patterns = [
        # Revocation patterns (complete revocation only - NOT partial)
        r'\brevogad[oa](?!\s*parcial)',        # Revoked but NOT partial
        r'\brevoga[rd](?!\s*parcial)',         # Revoke but NOT partial
        
        # Procedence patterns (complete only - NOT partial)
        r'\bprocedente(?!\s*(?:parcial|em\s*parte))',  # Procedent but NOT partial
        r'\btotalmente\s+procedente',          # Totally procedent
        
        # Appeal granted completely (NOT partial)
        r'\bprovido(?!\s*parcial)',            # Granted but NOT partial
        r'\bprovimento(?!\s*parcial)',         # Grant but NOT partial
        
        # Review granted (NOT partial)
        r'\bconcedid[oa](?!\s*parcial)',       # Granted but NOT partial
        
        # Decision altered
        r'\balterad[oa]',                      # Decision altered
        r'\balterar',                          # Alter verb forms
        
        # Conviction (in civil cases)
        r'\bcondenad[oa]',                     # Convicted
    ]
    
    # Try favorable patterns
    for pattern in favorable_patterns:
        if re.search(pattern, text_clean):
            return 'REVOGADA'
    
    # PRIORITY 3: Check for UNFAVORABLE patterns (explicitly exclude partials)
    unfavorable_patterns = [
        # Any rejection/denial (NOT partial)
        r'\bnegad[oa](?!\s*parcial)',          # Denied but NOT partial
        r'\bnega[rd](?!\s*parcial)',           # Deny but NOT partial
        
        # Any confirmation (maintains previous decision - NOT partial)
        r'\bconfirmad[oa](?!\s*parcial)',      # Confirmed but NOT partial
        r'\bconfirma[rd](?!\s*parcial)',       # Confirm but NOT partial
        r'\bconfirma[çc][aã]o(?!\s*parcial)',  # Confirmation but NOT partial
        r'\bmantid[oa](?!\s*parcial)',         # Maintained but NOT partial
        
        # Action/appeal denied (NOT partial)
        r'\bimprocedente(?!\s*(?:parcial|em\s*parte))',  # Improcedent but NOT partial
        r'\bimprocedência',                    # Improcedence
        
        # Specific appeal outcomes
        r'\bapela[cç][aã]o\s+improcedente',   # Appeal improcedent
        
        # Appeal denials (NOT partial)
        r'\bnegado\s+provimento(?!\s*parcial)',  # Appeal denied but NOT partial
        r'\brecurso.*improcedente',            # Appeal unsuccessful
    ]
    
    # Try unfavorable patterns
    for pattern in unfavorable_patterns:
        if re.search(pattern, text_clean):
            return 'NEGADO PROVIMENTO'
    
    return None


In [ ]:
df_class_fixed = df_fixed.copy()

# Filter that selects rows where 'decisao' is null or empty
mask = df_class_fixed['decisao'].isnull() | df_class_fixed['decisao'].astype(str).str.strip().eq('')

for index in df_class_fixed[mask].index:
    verbose_decision = df_class_fixed.at[index, 'decisao_extraida_do_texto_integral']
    summary_decision = extract_decision_summary(verbose_decision)
    # assign into decisao (and optionally keep decisao_resumo_extraida)
    df_class_fixed.at[index, 'decisao'] = summary_decision
    df_class_fixed.at[index, 'decisao_resumo_extraida'] = summary_decision
    print(index, summary_decision)

In [ ]:
df_class_fixed


In [ ]:
# Drop columns: 'url', 'tribunal' and 18 other columns
df_class = df_class_fixed.drop(columns=['url', 'tribunal', 'tipo_direito', 'tipo_caso', 'n_processo', 'juiz_relator', 'data_acordao' , 'sumario','votacao','meio_processual', 'texto_integral_disponivel', 
                                        'texto_integral_completo', 'decisao_extraida_do_texto_integral', 'metadata_decisao_extraction_method', 
                                        'metadata_decisao_confidence', 'metadata_decisao_keyword_found', 'metadata_decisao_requires_manual_review', 'metadata_decisao_review_reason', 
                                        'metadata_decisao_document_position', 'decisao_resumo_extraida'])

df_eda = df_class_fixed.drop(columns=['url', 'tipo_direito', 'tipo_caso', 'n_processo' , 'sumario','votacao','meio_processual', 'texto_integral_disponivel', 
                                        'texto_integral_completo', 'decisao_extraida_do_texto_integral', 'metadata_decisao_extraction_method', 
                                        'metadata_decisao_confidence', 'metadata_decisao_keyword_found', 'metadata_decisao_requires_manual_review', 'metadata_decisao_review_reason', 
                                        'metadata_decisao_document_position', 'decisao_resumo_extraida'])




# Deixar tbm descritores,full_text,

In [ ]:
df_class

In [ ]:
df_eda

In [ ]:
def extract_decision_binary_from_summary(summary_decision):
    """
    
    MELHOR ATE AGORA
    
    
    Classifica decisões sumárias em binário com lógica "tudo ou nada":
    - FAVORÁVEL: Decisão completamente positiva para o recorrente
    - DESFAVORÁVEL: Qualquer coisa que não seja vitória completa
    
    REGRA: Parciais = DESFAVORÁVEL (recorrente não obteve tudo)
    """
    if not summary_decision or pd.isna(summary_decision):
        return None
    
    # Convert to lowercase and clean
    text = str(summary_decision).lower().strip()
    # Remove punctuation for better matching
    text_clean = re.sub(r'[^\w\s]', ' ', text)
    
    # Define STRICTLY FAVORABLE patterns (complete victory only)
    strictly_favorable_patterns = [
        # Revocation patterns (any form of complete revocation)
        r'\brevogad[oa](?!\s*parcial)',     # Revoked but NOT partial
        r'\brevoga[rd](?!\s*parcial)',      # Revoke but NOT partial
        
        # Procedence patterns (complete only)
        r'\bprocedente(?!\s*(?:parcial|em\s*parte))',  # Procedent but NOT partial
        
        # Appeal granted patterns
        r'\bprovido(?!\s*parcial)',         # Granted but NOT partial
        r'\bprovimento(?!\s*parcial)',      # Grant but NOT partial
        
        # Review/revista granted patterns  
        r'\bconcedid[oa](?!\s*parcial)',    # Granted but NOT partial
        
        # Decision altered patterns
        r'\balterad[oa]',                   # Decision altered
        r'\balterar',                  # Alter verb forms
        
        # Conviction patterns (in civil cases, usually favorable)
        r'\bcondenad[oa]',                  # Convicted
        
       
    ]
    
    # Everything else is UNFAVORABLE (including partials and all rejections)
    unfavorable_patterns = [
        # Any rejection/denial
        r'\bnegad[oa]',                     # Any form of denied
        r'\bnega[rd]',                      # Deny verb forms
        
        # Any confirmation (maintains previous unfavorable decision)
        r'\bconfirmad[oa]',                 # Any form of confirmed
        r'\bconfirma[rd]',                  # Confirm verb forms
        r'\bconfirma[çc][aã]o',            # Confirmation noun
        r'\bmantid[oa]',                    # Maintained
        
        # Action/appeal denied
        r'\bimprocedente(?!\s*(?:a\s*)?apela[cç][aã]o)', # Improcedent but NOT "appeal improcedent"
        r'\bimprocedência',                 # Improcedence
        
         # Specific appeal outcomes that indicate action lost
        r'\bapela[cç][aã]o\s+improcedente', # Appeal improcedent 
        r'\bimprocedente\s+a\s+apela[cç][aã]o', # Appeal improcedent
        
        # Any partial outcome (treated as unfavorable)
        r'\bparcial',                       # Any partial
        r'\bparcialmente',                  # Partially
        r'\bem\s+parte',                    # In part
        
        # Appeal denials
        r'\bnegado\s+provimento',           # Appeal denied
        r'\brecurso.*improcedente',         # Appeal unsuccessful
    ]
    
    # Check for STRICTLY favorable first (must be complete victory)
    for pattern in strictly_favorable_patterns:
        if re.search(pattern, text_clean):
            return 'FAVORÁVEL'
    
    # Then check for unfavorable patterns
    for pattern in unfavorable_patterns:
        if re.search(pattern, text_clean):
            return 'DESFAVORÁVEL'
    
    # Default: if we can't classify clearly, assume unfavorable (conservative approach)
    return None



In [ ]:
def extract_decision_ternary_from_summary(summary_decision):
    """
    Classifica decisões sumárias em três categorias:
    - TOTALMENTE FAVORÁVEL: Decisão completamente positiva para o recorrente
    - TOTALMENTE DESFAVORÁVEL: Decisão completamente negativa para o recorrente
    - PARCIAL: Decisão parcialmente favorável/desfavorável
    
    REGRA: Identifica explicitamente decisões parciais como categoria própria
    """
    if not summary_decision or pd.isna(summary_decision):
        return None
    
    # Convert to lowercase and clean
    text = str(summary_decision).lower().strip()
    # Remove punctuation for better matching
    text_clean = re.sub(r'[^\w\s]', ' ', text)
    
    # Define PARTIAL patterns (check first - highest priority)
    partial_patterns = [
        r'\bparcial',                           # Partial
        r'\bparcialmente',                      # Partially
        r'\bem\s+parte',                        # In part
        r'\brevogad[oa]\s+parcial',            # Partially revoked
        r'\bprocedente\s+parcial',             # Partially procedent
        r'\bprovido\s+parcial',                # Partially granted
        r'\bconcedid[oa]\s+parcial',           # Partially granted
        r'\bconfirmad[oa]\s+parcial',          # Partially confirmed
    ]
    
    # Define TOTALLY FAVORABLE patterns
    totally_favorable_patterns = [
        # Complete revocation
        r'\brevogad[oa](?!\s*parcial)',        # Revoked but NOT partial
        r'\brevoga[rd](?!\s*parcial)',         # Revoke but NOT partial
        
        # Complete procedence
        r'\bprocedente(?!\s*(?:parcial|em\s*parte))',  # Procedent but NOT partial
        r'\btotalmente\s+procedente',          # Totally procedent
        
        # Appeal granted completely
        r'\bprovido(?!\s*parcial)',            # Granted but NOT partial
        r'\bprovimento(?!\s*parcial)',         # Grant but NOT partial
        
        # Review granted
        r'\bconcedid[oa](?!\s*parcial)',       # Granted but NOT partial
        
        # Decision altered
        r'\balterad[oa]',                      # Decision altered
        r'\balterar',                          # Alter verb forms
        
        # Conviction (in civil cases)
        r'\bcondenad[oa]',                     # Convicted
    ]
    
    # Define TOTALLY UNFAVORABLE patterns
    totally_unfavorable_patterns = [
        # Complete rejection/denial
        r'\bnegad[oa](?!\s*parcial)',          # Denied but NOT partial
        r'\bnega[rd](?!\s*parcial)',           # Deny but NOT partial
        
        # Complete confirmation (maintains previous decision)
        r'\bconfirmad[oa](?!\s*parcial)',      # Confirmed but NOT partial
        # Confirm but NOT partial
         r'\bconfirma[çc][aã]o(?!\s*parcial)',  # Confirmation noun but NOT partial
         r'\bmantid[oa](?!\s*parcial)',         # Maintained but NOT partial
         
         # Action/appeal denied
         r'\bimprocedente(?!\s*(?:(?:a\s*)?apela[cç][aã]o|parcial|em\s*parte))',  # Improcedent but NOT appeal or partial
         r'\bimprocedência(?!\s*parcial)',      # Improcedence but NOT partial
         r'\btotalmente\s+improcedente',        # Totally improcedent
         
         # Specific appeal outcomes
         r'\bapela[cç][aã]o\s+improcedente',    # Appeal improcedent 
         r'\bimprocedente\s+a\s+apela[cç][aã]o', # Appeal improcedent
         
         # Appeal denials
         r'\bnegado\s+provimento',              # Appeal denied
         r'\brecurso.*improcedente',            # Appeal unsuccessful
         
         # Absolution (in some contexts unfavorable for plaintiff)
         r'\babsolvid[oa](?!\s*parcial)',       # Absolved but NOT partial
    ]
    
    # Check for PARTIAL first (highest priority)
    for pattern in partial_patterns:
        if re.search(pattern, text_clean):
            return 'PARCIAL'
    
    # Then check for TOTALLY FAVORABLE
    for pattern in totally_favorable_patterns:
        if re.search(pattern, text_clean):
            return 'TOTALMENTE FAVORÁVEL'
    
    # Finally check for TOTALLY UNFAVORABLE
    for pattern in totally_unfavorable_patterns:
        if re.search(pattern, text_clean):
            return 'TOTALMENTE DESFAVORÁVEL'
    
    # Default: if we can't classify clearly, return None
    return None

In [ ]:
df_bin_class = df_class.copy()
df_bin_class['decisao_binaria'] = df_bin_class['decisao'].apply(extract_decision_binary_from_summary)
df_bin_class = df_bin_class.dropna(subset=['decisao_binaria'])

df_bin_eda = df_eda.copy()
df_bin_eda['decisao_binaria'] = df_bin_eda['decisao'].apply(extract_decision_binary_from_summary)
df_bin_eda = df_bin_eda.dropna(subset=['decisao_binaria'])


df_final_bin_class = df_bin_class.copy() # Falta depois o encoding das decisões binárias
df_final_bin_eda = df_bin_eda.copy() 


In [ ]:
df_final_bin_class

In [ ]:
df_final_bin_eda

In [ ]:
# # Gravar os dataframes finais em CSV
# df_final_bin_class.to_csv(Path(f'./../../data/processed_data/classification/binary/df_acordaos_{type_case}_classification_binary.csv'), index=False)
# df_final_bin_eda.to_csv(Path(f'./../../data/processed_data/eda/binary/df_acordaos_{type_case}_eda_binary.csv'), index=False)

In [ ]:
df_ter_class = df_class.copy()
df_ter_class['decisao_ternaria'] = df_ter_class['decisao'].apply(extract_decision_ternary_from_summary)
df_ter_class = df_ter_class.dropna(subset=['decisao_ternaria'])


df_ter_eda = df_eda.copy()
df_ter_eda['decisao_ternaria'] = df_ter_eda['decisao'].apply(extract_decision_ternary_from_summary)
df_ter_eda = df_ter_eda.dropna(subset=['decisao_ternaria'])

df_final_ter_class = df_ter_class.copy() # Falta depois o encoding das decisões binárias
df_final_ter_eda = df_ter_eda.copy() 



In [ ]:
df_final_ter_class

In [ ]:
df_final_ter_eda

In [ ]:
# df_final_ter_class.to_csv(Path(f'./../../data/processed_data/classification/ternary/df_acordaos_{type_case}_classification_ternary.csv'), index=False)
# df_final_ter_eda.to_csv(Path(f'./../../data/processed_data/eda/ternary/df_acordaos_{type_case}_eda_ternary.csv'), index=False)

In [ ]:
# Count DESFAVORÁVEL in binary classification
desfavoravel_bin = (df_final_bin_class['decisao_binaria'] == 'DESFAVORÁVEL').sum()

# Count TOTALMENTE DESFAVORÁVEL in ternary classification
desfavoravel_ter = (df_final_ter_class['decisao_ternaria'] == 'TOTALMENTE DESFAVORÁVEL').sum()

# Count PARCIAL in ternary classification
parcial_ter = (df_final_ter_class['decisao_ternaria'] == 'PARCIAL').sum()

print(f"DESFAVORÁVEL (binary): {desfavoravel_bin}")
print(f"TOTALMENTE DESFAVORÁVEL (ternary): {desfavoravel_ter}")
print(f"PARCIAL (ternary): {parcial_ter}")
print(f"Sum of TOTALMENTE DESFAVORÁVEL + PARCIAL: {desfavoravel_ter + parcial_ter}")
print(f"\nDifference: {desfavoravel_bin - (desfavoravel_ter + parcial_ter)}")

## Preparação Classificação

USANDO df_final_bin_class e df_final_ter_class

- dividing train_test split (trying to not imbalance classes too much using stratified split)
on training split:

- stopwords removal. If possible legal stop words removal (not mandatory)
- lematization
- law articles detection, removal from main text  and storage outside the training data but correcly indexed to its specific case.
-  feature engineering (baseline can be tf-idf), word2vec or doc2vec, then mix of both
-  - dealing with class imbalance if necessary 

## Preparação EDA

USANDO df_final_bin_eda e df_final_ter_eda

- Basics Stats like distribution of dates, length of text,words and sentences without decision, length distribution per class, classes distribution. word(n-grams) frequence per class (after stopwords removal) - DONE
  
- Descriptors frequency (outside the main ones, e.g 'VIOLÊNCIA DOMÉSTICA', 'INCUMPRIMENTO DO/DE CONTRATO/CONTRATUAL).
- Trafnorming judge names to gender (male, female, non descriptive(meaning we cant extract gender by the names, for example can be just surnames)). Then get class distribution based on gender
- Class distribution based on Tribunal
- sentiment analysis (to understand if domestic violence cases are in fact more emotional than contract breaches or if there is no signigicant difference)
- flesch_reading_ease para comparar a dificuldade de leitura entre casos de violencia domestica vs contract breach, perceber se a diferença é signigicativa ou não.



In [ ]:
import pandas as pd
import seaborn as sb
import re

type_case = 'ic'  # 'dv' para Violência Doméstica, 'ic' para Incumprimento de Contratos
type_case_verbose = 'Contract breach' if type_case == 'ic' else 'Domestic Violence'
type_case_folder = 'contract_breach' if type_case == 'ic' else 'domestic_violence'
type_analysis = 'eda'  # 'classification' ou 'eda'
df_bin_eda = pd.read_csv(f'./../../data/processed_data/eda/binary/df_acordaos_{type_case}_{type_analysis}_binary.csv')
df_ter_eda = pd.read_csv(f'./../../data/processed_data/eda/ternary/df_acordaos_{type_case}_{type_analysis}_ternary.csv')

In [ ]:
df_bin_eda

In [ ]:
df_ter_eda

### Basic Statistics and Viz

In [ ]:
import seaborn as sb

import matplotlib.pyplot as plt

# Create figure with 2 subplots for both binary and ternary
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Binary classification - class distribution
class_counts_bin = df_bin_eda['decisao_binaria'].value_counts()
sb.barplot(x=class_counts_bin.index, y=class_counts_bin.values, ax=axes[0], palette='bright')
axes[0].set_title(f'Binary Classification - Class Distribution\n{type_case_verbose} cases', 
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel('Decision', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].grid(axis='y', alpha=0.3)

# Add percentage labels on bars
for i, (label, count) in enumerate(class_counts_bin.items()):
    percentage = (count / len(df_bin_eda)) * 100
    # Position label at center of bar
    axes[0].text(i, count/2, f'{count}\n({percentage:.1f}%)', 
                ha='center', va='center', fontsize=11, fontweight='bold',
                color='white',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='red', 
                         edgecolor='darkred', alpha=0.8))

# Ternary classification - class distribution
class_counts_ter = df_ter_eda['decisao_ternaria'].value_counts()
sb.barplot(x=class_counts_ter.index, y=class_counts_ter.values, ax=axes[1], palette='bright')
axes[1].set_title(f'Ternary Classification - Class Distribution\n{type_case_verbose} cases', 
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Decision', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_xticklabels(class_counts_ter.index, rotation=15, ha='right')
axes[1].grid(axis='y', alpha=0.3)

# Add percentage labels on bars
for i, (label, count) in enumerate(class_counts_ter.items()):
    percentage = (count / len(df_ter_eda)) * 100
    axes[1].text(i, count/2, f'{count}\n({percentage:.1f}%)', 
                ha='center', va='center', fontsize=11, fontweight='bold',
                color='white',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='red', 
                         edgecolor='darkred', alpha=0.8))

plt.tight_layout()
# plt.savefig(Path(f'./eda_viz/{type_case_folder}/basic_stats/class_distribution_{type_case}.png'), 
#            bbox_inches='tight', dpi=300)
# plt.show()

# Print summary statistics
print(f"\n{'='*60}")
print(f"CLASS DISTRIBUTION SUMMARY - {type_case_verbose.upper()}")
print(f"{'='*60}\n")

print("BINARY CLASSIFICATION:")
for label, count in class_counts_bin.items():
    percentage = (count / len(df_bin_eda)) * 100
    print(f"  {label:20s}: {count:4d} ({percentage:5.1f}%)")

print(f"\nTERNARY CLASSIFICATION:")
for label, count in class_counts_ter.items():
    percentage = (count / len(df_ter_eda)) * 100
    print(f"  {label:25s}: {count:4d} ({percentage:5.1f}%)")

In [ ]:
# First I will remove all punctuation and special carachters from texto_sem_decisao and put all in lower case.

import re

def clean_text(text):
    
    text = str(text)
    
     # Check if text is a string
    if not isinstance(text,str):
        return ""
    
    # Lowecasing
    text = text.lower()
    
    # Remove punctuation and special characters using regex
    text = re.sub(r'[^\w\s]', ' ', text)
    
    return text
    

In [ ]:
df_bin_eda['clean_text'] = df_bin_eda['texto_integral_sem_decisao'].apply(clean_text)
df_ter_eda['clean_text'] = df_ter_eda['texto_integral_sem_decisao'].apply(clean_text)

# df_bin_eda.drop(columns=['texto_integral_sem_decisao'], axis=1, inplace=True)
# df_ter_eda.drop(columns=['texto_integral_sem_decisao'], axis=1, inplace=True)

#Punctuation and special carachters dont add relevant info for eda.

In [ ]:
# df_bin_eda['clean_text'] 

In [ ]:
# Counting caracthers in clean text column

df_bin_eda['text_length_clean'] = df_bin_eda['clean_text'].apply(len)
df_ter_eda['text_length_clean'] = df_ter_eda['clean_text'].apply(len)

mean_length_clean = df_bin_eda['text_length_clean'].mean()

# If 1 token ~ 4 carachters, then mean tokens is mean_length/4 source:https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them

print(f'Mean text length: {round(mean_length_clean)} characters - CLEAN TEXT')
print(f'Mean text length: {round(mean_length_clean/4)} tokens (approx.) - CLEAN TEXT')

In [ ]:
# Counting caracthers in raw text column

df_bin_eda['text_length_raw'] = df_bin_eda['texto_integral_sem_decisao'].apply(len)
df_ter_eda['text_length_raw'] = df_ter_eda['texto_integral_sem_decisao'].apply(len)

mean_length_raw = df_bin_eda['text_length_raw'].mean()

# If 1 token ~ 4 carachters, then mean tokens is mean_length/4 source:https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them

print(f'Mean text length: {round(mean_length_raw)} characters - RAW TEXT')
print(f'Mean text length: {round(mean_length_raw/4)} tokens (approx.) - RAW TEXT')

In [ ]:
# Counting word and sentences in clean text column

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
# nltk.download('punkt')
# nltk.download('punkt_tab')

df_bin_eda['word_count'] = df_bin_eda['clean_text'].apply(lambda x: len(word_tokenize(x,language='portuguese')))
df_bin_eda['sentence_count'] = df_bin_eda['texto_integral_sem_decisao'].apply(lambda x: len(sent_tokenize(x,language='portuguese')))

mean_word_count = df_bin_eda['word_count'].mean()
mean_sentence_count = df_bin_eda['sentence_count'].mean()

print(f'Mean word count: {round(mean_word_count)} words')
print(f'Mean sentence count: {round(mean_sentence_count)} sentences')  # since we removed punctuation, sentence count is not accurate



In [ ]:
# Extra cleaning to remove noise to improve interpretation of EDA results

def clean_text_advanced(text):
    """
    Advanced cleaning: removes numbers, single characters, and extra whitespace
    Use AFTER basic clean_text() or on raw text
    """
    text = str(text)
    
    if not isinstance(text, str):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # Remove standalone numbers (including those with º like "1º", "2º")
    text = re.sub(r'\b\d+º?\b', ' ', text)  # Matches "1", "2º", "123", etc.
    
    # Remove single characters (but keep words)
    text = re.sub(r'\b\w\b', ' ', text)  # Removes "n", "º", "a" when standalone
    
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [ ]:
df_bin_eda['clean_text'] = df_bin_eda['texto_integral_sem_decisao'].apply(clean_text_advanced)
df_ter_eda['clean_text'] = df_ter_eda['texto_integral_sem_decisao'].apply(clean_text_advanced)


In [ ]:
# nltk.download('stopwords')
from nltk.probability import FreqDist
stopwords = set(nltk.corpus.stopwords.words('portuguese'))

# removing stopwords

df_bin_eda['tokens_no_stopwords'] = df_bin_eda['clean_text'].apply(lambda x: [word for word in word_tokenize(x) if word not in stopwords])

# Creating a frequency distribution of words in the corpus without stopwords

most_common_words_bin = FreqDist()
for tokens in df_bin_eda['tokens_no_stopwords']:
    most_common_words_bin.update(tokens)
    
    
most_common_words_bin = most_common_words_bin.most_common(20)
word_bin,freq_bin = [],[]
print("Top 20 most common words (without stopwords) - Binary cases:")
for word,freq in most_common_words_bin:
    word_bin.append(word)
    freq_bin.append(freq)
    
    print(f'{word} - Frequency: {freq}')
    
    


    
    

In [ ]:
# removing stopwords

df_ter_eda['tokens_no_stopwords'] = df_ter_eda['clean_text'].apply(lambda x: [word for word in word_tokenize(x) if word not in stopwords])

# Creating a frequency distribution of words in the corpus without stopwords

most_common_words_ter = FreqDist()
for tokens in df_bin_eda['tokens_no_stopwords']:
    most_common_words_ter.update(tokens)
    
    
most_common_words_ter = most_common_words_ter.most_common(20)
word_ter,freq_ter = [],[]
print("Top 20 most common words (without stopwords) - Ternary cases:")
for word,freq in most_common_words_ter:
    word_ter.append(word)
    freq_ter.append(freq)
    
    print(f'{word} - Frequency: {freq}')

In [ ]:
from nltk.probability import FreqDist
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from pathlib import Path

def get_most_common_words_per_class(df, class_column, text_column='tokens_no_stopwords', top_n=10,distinct=False):
    """
    Get most common words for each class in the dataframe.
    
    Args:
        df: DataFrame with tokenized text
        class_column: Name of the column containing class labels
        text_column: Name of column containing tokenized text (default: 'tokens_no_stopwords')
        top_n: Number of most common words to return (default: 10)
    
    Returns:
        dict: Dictionary with class labels as keys and list of (word, frequency) tuples as values
    """
    results = {}
   
    
    # Get unique classes
    classes = df[class_column].unique()
    
    for class_label in classes:
        # Filter dataframe for this class
        df_class = df[df[class_column] == class_label]
        
        # Create frequency distribution for this class
        freq_dist = FreqDist()
        for tokens in df_class[text_column]:
            if isinstance(tokens, list):
                freq_dist.update(tokens)
        
        # Get most common words
        most_common = freq_dist.most_common(top_n)
        results[class_label] = most_common
        
        # Print results
        print(f"\n{'='*60}")
        print(f"Class: {class_label}")
        print(f"{'='*60}")
        for word, freq in most_common:
            print(f"  {word:20s} - {freq:6d}")
    
    return results

def plot_most_common_words_per_class(results, class_column, title="Most Common Words by Class", distinct=False):
    """
    Create bar plots for most common words per class.
    
    Args:
        results: Dictionary from get_most_common_words_per_class()
        class_column: Name of the class column (for saving file)
        title: Title for the plot
        distinct: Boolean indicating if words are distinct per class
    """
    
    num_classes = len(results)
    fig, axes = plt.subplots(1, num_classes, figsize=(8*num_classes, 6))
    
    # Handle single class case
    if num_classes == 1:
        axes = [axes]
    
    for idx, (class_label, word_freq_list) in enumerate(results.items()):
        words = [word for word, freq in word_freq_list]
        freqs = [freq for word, freq in word_freq_list]
        
        sb.barplot(x=freqs, y=words, ax=axes[idx], palette='viridis')
        axes[idx].set_title(f'Class: {class_label}')
        axes[idx].set_xlabel('Frequency')
        axes[idx].set_ylabel('Word')
    
    plt.suptitle(title, fontsize=16, y=1.02)
    plt.tight_layout()
    
    # Save the figure before showing it
    class_column_lower = class_column.lower()
    # if distinct:
    #     plt.savefig(Path(f'./eda_viz/{type_case_folder}/basic_stats/most_common_words_{type_case}_{class_column_lower}_distinct.png'), bbox_inches='tight', dpi=300)
    # else:
    #     plt.savefig(Path(f'./eda_viz/{type_case_folder}/basic_stats/most_common_words_{type_case}_{class_column_lower}.png'), bbox_inches='tight', dpi=300)
    
    plt.show()






In [ ]:
### PERCEBER MELHOR A FUNÇÃO ABAIXO ###

def get_distinctive_words_per_class(df, class_column, text_column='tokens_no_stopwords', top_n=10):
    """
    Get words that are most distinctive for each class using TF-IDF from sklearn.
    Words that appear frequently in one class but rarely in others.
    
    Args:
        df: DataFrame with tokenized text
        class_column: Name of the column containing class labels
        text_column: Name of column containing tokenized text
        top_n: Number of most distinctive words to return
    
    Returns:
        dict: Dictionary with class labels as keys and list of (word, score) tuples as values
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    import numpy as np
    
    results = {}
    classes = df[class_column].unique()
    
    # Prepare documents per class (join tokens back into strings)
    class_documents = {}
    for class_label in classes:
        df_class = df[df[class_column] == class_label]
        # Join tokens back into space-separated strings
        documents = df_class[text_column].apply(
            lambda tokens: ' '.join(tokens) if isinstance(tokens, list) else ''
        ).tolist()
        class_documents[class_label] = documents
    
    # Calculate TF-IDF for each class
    for class_label in classes:
        # Fit TF-IDF on this class's documents
        vectorizer = TfidfVectorizer(
            max_features=None,
            lowercase=False,  # Already lowercased
            token_pattern=r'(?u)\b\w+\b'  # Match words
        )
        
        try:
            tfidf_matrix = vectorizer.fit_transform(class_documents[class_label])
            feature_names = vectorizer.get_feature_names_out()
            
            # Calculate mean TF-IDF score for each word across all documents in this class
            mean_tfidf_scores = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
            
            # Create word-score pairs
            word_scores = list(zip(feature_names, mean_tfidf_scores))
            
            # Sort by score (descending) and get top N
            distinctive_words = sorted(word_scores, key=lambda x: x[1], reverse=True)[:top_n]
            
            results[class_label] = distinctive_words
            
            # Print results
            print(f"\n{'='*60}")
            print(f"Distinctive words for: {class_label}")
            print(f"{'='*60}")
            for word, score in distinctive_words:
                print(f"  {word:20s} - TF-IDF: {score:8.6f}")
                
        except Exception as e:
            print(f"Error processing class {class_label}: {e}")
            results[class_label] = []
    
    return results

def plot_most_tf_idf_per_class(results, class_column, title="Most Distinctive Words by TF-IDF per Class", distinct=False):
    """
    Create bar plots for most distinctive words per class based on TF-IDF scores.
    
    Args:
        results: Dictionary from get_distinctive_words_per_class()
        class_column: Name of the class column (for saving file)
        title: Title for the plot
        distinct: Boolean indicating if words are distinct per class
    """
    
    num_classes = len(results)
    fig, axes = plt.subplots(1, num_classes, figsize=(8*num_classes, 6))
    
    # Handle single class case
    if num_classes == 1:
        axes = [axes]
    
    for idx, (class_label, word_score_list) in enumerate(results.items()):
        words = [word for word, score in word_score_list]
        scores = [score for word, score in word_score_list]
        
        sb.barplot(x=scores, y=words, ax=axes[idx], palette='viridis')
        axes[idx].set_title(f'Class: {class_label}', fontsize=14, fontweight='bold')
        axes[idx].set_xlabel('Mean TF-IDF Score', fontsize=12)
        axes[idx].set_ylabel('Word', fontsize=12)
    
    plt.suptitle(title, fontsize=16, y=1.02)
    plt.tight_layout()
    
    # Save the figure before showing it
    class_column_lower = class_column.lower()
    # if distinct:
    #     plt.savefig(Path(f'./eda_viz/{type_case_folder}/basic_stats/distinctive_words_tfidf_{type_case}_{class_column_lower}_distinct.png'), bbox_inches='tight', dpi=300)
    # else:
    #     plt.savefig(Path(f'./eda_viz/{type_case_folder}/basic_stats/distinctive_words_tfidf_{type_case}_{class_column_lower}.png'), bbox_inches='tight', dpi=300)
    
    plt.show()


In [ ]:
from nltk.util import ngrams

def get_ngram_frequency_per_class(df, class_column, text_column='tokens_no_stopwords', 
                                   n=2, top_n=20):
    """
    Get most common n-grams for each class.
    
    Args:
        df: DataFrame with tokenized text
        class_column: Name of the column containing class labels
        text_column: Name of column containing tokenized text
        n: N-gram size (2=bigrams, 3=trigrams, etc.)
        top_n: Number of most common n-grams to return
    
    Returns:
        dict: Dictionary with class labels as keys and list of (ngram, frequency) tuples
    """
    results = {}
    classes = df[class_column].unique()
    
    for class_label in classes:
        # Filter dataframe for this class
        df_class = df[df[class_column] == class_label]
        
        # Create frequency distribution for n-grams
        freq_dist = FreqDist()
        
        for tokens in df_class[text_column]:
            if isinstance(tokens, list) and len(tokens) >= n:
                # Generate n-grams from token list
                ngram_list = list(ngrams(tokens, n))
                # Join tuples into strings for better readability
                ngram_strings = [' '.join(gram) for gram in ngram_list]
                freq_dist.update(ngram_strings)
        
        # Get most common n-grams
        most_common = freq_dist.most_common(top_n)
        results[class_label] = most_common
        
        # Print results
        ngram_name = {1: "unigrams", 2: "bigrams", 3: "trigrams", 4: "4-grams"}.get(n, f"{n}-grams")
        print(f"\n{'='*70}")
        print(f"Class: {class_label} - Top {top_n} {ngram_name}")
        print(f"{'='*70}")
        for ngram, freq in most_common:
            print(f"  {ngram:45s} - {freq:6d}")
    
    return results


def get_tfidf_per_class(df, class_column, text_column='tokens_no_stopwords', 
                        ngram_range=(1, 1), top_n=20):
    """
    Get TF-IDF scores for each class (NOT class-vs-rest, just within-class TF-IDF).
    
    Args:
        df: DataFrame with tokenized text
        class_column: Name of column containing class labels
        text_column: Name of column containing tokenized text
        ngram_range: Tuple (min_n, max_n) for n-grams (1,1)=unigrams, (1,2)=uni+bigrams, (2,2)=bigrams only
        top_n: Number of top TF-IDF terms to return
    
    Returns:
        dict: Dictionary with class labels as keys and list of (term, tfidf_score) tuples
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    import numpy as np
    
    results = {}
    classes = df[class_column].unique()
    
    for class_label in classes:
        # Filter dataframe for this class
        df_class = df[df[class_column] == class_label]
        
        # Convert token lists to strings
        documents = df_class[text_column].apply(
            lambda tokens: ' '.join(tokens) if isinstance(tokens, list) else ''
        ).tolist()
        
        # Calculate TF-IDF within this class
        vectorizer = TfidfVectorizer(
            ngram_range=ngram_range,
            max_features=None,
            lowercase=False,  # Already lowercased
            token_pattern=r'(?u)\b\w+\b'
        )
        
        try:
            tfidf_matrix = vectorizer.fit_transform(documents)
            feature_names = vectorizer.get_feature_names_out()
            
            # Calculate mean TF-IDF across all documents in this class
            mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
            
            # Create term-score pairs
            term_scores = list(zip(feature_names, mean_tfidf))
            
            # Sort by TF-IDF score and get top N
            top_terms = sorted(term_scores, key=lambda x: x[1], reverse=True)[:top_n]
            
            results[class_label] = top_terms
            
            # Print results
            ngram_desc = {(1,1): "unigrams", (1,2): "unigrams + bigrams", 
                         (2,2): "bigrams", (3,3): "trigrams"}.get(ngram_range, f"{ngram_range}")
            print(f"\n{'='*70}")
            print(f"Class: {class_label} - Top {top_n} TF-IDF {ngram_desc}")
            print(f"{'='*70}")
            for term, score in top_terms:
                print(f"  {term:45s} - {score:8.6f}")
        
        except Exception as e:
            print(f"Error processing class {class_label}: {e}")
            results[class_label] = []
    
    return results


def plot_ngram_frequency_per_class(results, title="N-gram Frequency by Class", n=2, class_column='decisao_ternaria'):
    """
    Create horizontal bar plots for n-gram frequencies per class.
    
    Args:
        results: Dictionary from get_ngram_frequency_per_class()
        title: Title for the plot
        n: N-gram size (for labeling)
    """
    num_classes = len(results)
    fig, axes = plt.subplots(1, num_classes, figsize=(10*num_classes, 8))
    
    # Handle single class case
    if num_classes == 1:
        axes = [axes]
    
    for idx, (class_label, ngram_freq_list) in enumerate(results.items()):
        ngrams_text = [ngram for ngram, freq in ngram_freq_list]
        freqs = [freq for ngram, freq in ngram_freq_list]
        
        sb.barplot(x=freqs, y=ngrams_text, ax=axes[idx], palette='viridis')
        axes[idx].set_title(f'Class: {class_label}', fontsize=14, fontweight='bold')
        axes[idx].set_xlabel('Frequency', fontsize=12)
        axes[idx].set_ylabel('N-gram', fontsize=12)
        axes[idx].tick_params(axis='y', labelsize=9)
    
    plt.suptitle(title, fontsize=16, y=1.00, fontweight='bold')
    plt.tight_layout()
    
    if n == 2:
        ngram_name = 'bigrams'
    elif n == 3:
        ngram_name = 'trigrams'
        
    elif n == 4:
        ngram_name = '4-grams'
    
    elif n == 5:
        ngram_name = '5-grams'
        
        
    plt.savefig(Path(f'./eda_viz/{type_case_folder}/basic_stats/{ngram_name}_frequency_{type_case}_{class_column}.png'), 
            bbox_inches='tight', dpi=300)
    
    
    plt.show()


def plot_tfidf_per_class(results, title="TF-IDF Scores by Class", n=2, class_column='decisao_ternaria'):
    """
    Create horizontal bar plots for TF-IDF scores per class.
    
    Args:
        results: Dictionary from get_tfidf_per_class()
        title: Title for the plot
    """
    num_classes = len(results)
    fig, axes = plt.subplots(1, num_classes, figsize=(10*num_classes, 8))
    
    # Handle single class case
    if num_classes == 1:
        axes = [axes]
    
    for idx, (class_label, term_score_list) in enumerate(results.items()):
        terms = [term for term, score in term_score_list]
        scores = [score for term, score in term_score_list]
        
        sb.barplot(x=scores, y=terms, ax=axes[idx], palette='magma')
        axes[idx].set_title(f'Class: {class_label}', fontsize=14, fontweight='bold')
        axes[idx].set_xlabel('Mean TF-IDF Score', fontsize=12)
        axes[idx].set_ylabel('Term', fontsize=12)
        axes[idx].tick_params(axis='y', labelsize=9)
    
    plt.suptitle(title, fontsize=16, y=1.00, fontweight='bold')
    plt.tight_layout()
    
    if n == 2:
        ngram_name = 'bigrams'
    elif n == 3:
        ngram_name = 'trigrams'
        
    elif n == 4:
        ngram_name = '4-grams'
    
    elif n == 5:
        ngram_name = '5-grams'
        
    # plt.savefig(Path(f'./eda_viz/{type_case_folder}/basic_stats/{ngram_name}_tfidf_{type_case}_{class_column}.png'), 
    #         bbox_inches='tight', dpi=300)
    
    
    plt.show()

##### Uni,Bi,Tri;4,5 Grams

In [ ]:
# Apply to binary classification
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Top 20 Most Common Words Per Class")
print("="*80)
results_bin = get_most_common_words_per_class(
    df_bin_eda, 
    class_column='decisao_binaria',
    top_n=20
)
# Apply to ternary classification
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Top 20 Most Common Words Per Class")
print("="*80)
results_ter = get_most_common_words_per_class(
    df_ter_eda, 
    class_column='decisao_ternaria',
    top_n=20
)

plot_most_common_words_per_class(
    results_bin, 
    title=f"Top 20 Most Common Words by Binary Decision - {type_case_verbose} cases",
    class_column='decisao_binaria'
)



plot_most_common_words_per_class(
    results_ter, 
    title=f"Top 20 Most Common Words by Ternary Decision - {type_case_verbose} cases",
    class_column='decisao_ternaria'
)

In [ ]:


# Get distinctive words for binary classification
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Top 20 Most Distinctive Words Per Class")
print("="*80)
distinctive_bin = get_distinctive_words_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    top_n=20
)

# Get distinctive words for ternary classification
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Top 20 Most Distinctive Words Per Class")
print("="*80)
distinctive_ter = get_distinctive_words_per_class(
    df_ter_eda,
    class_column='decisao_ternaria',
    top_n=20

)


plot_most_tf_idf_per_class(
    distinctive_bin, 
    title=f"Top 20 TF-IDF Words by Binary Decision - {type_case_verbose} cases",
    class_column='decisao_binaria'
)

plot_most_tf_idf_per_class(
    distinctive_ter, 
    title=f"Top 20 TF-IDF Words by Ternary Decision - {type_case_verbose} cases",
    class_column='decisao_ternaria'
)

In [ ]:
#===================Binary classification ===================
# ================== BIGRAMS FREQUENCY ==================
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Top 20 Bigrams Per Class")
print("="*80)
bigrams_bin = get_ngram_frequency_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    n=2,
    top_n=20
)

plot_ngram_frequency_per_class(
    bigrams_bin,
    title=f"Top 20 Bigrams by Binary Decision - {type_case_verbose} cases ",
    n=2,
    class_column='decisao_binaria'
)



In [ ]:
# ================== TRIGRAMS FREQUENCY ==================
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Top 20 Trigrams Per Class")
print("="*80)
trigrams_bin = get_ngram_frequency_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    n=3,
    top_n=20
)

plot_ngram_frequency_per_class(
    trigrams_bin,
    title=f"Top 20 Trigrams by Binary Decision - {type_case_verbose} cases",
    n=3,
    class_column='decisao_binaria'
)



In [ ]:
# ================== TF-IDF BIGRAMS ==================
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Top 20 TF-IDF Bigrams Per Class")
print("="*80)
tfidf_bigrams_bin = get_tfidf_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    ngram_range=(2, 2),
    top_n=20
)

plot_tfidf_per_class(
    tfidf_bigrams_bin,
    title=f"Top 20 TF-IDF Bigrams by Binary Decision - {type_case_verbose} cases",
    n=2,
    class_column='decisao_binaria'
)



In [ ]:
#================= TF-IDF TRIGRAMS ==================
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Top 20 TF-IDF Trigrams Per Class")
print("="*80)
tfidf_trigrams_bin = get_tfidf_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    ngram_range=(3, 3),
    top_n=20
)

plot_tfidf_per_class(
    tfidf_trigrams_bin,
    title=f"Top 20 TF-IDF Trigrams by Binary Decision - {type_case_verbose} cases",
    n=3,
    class_column='decisao_binaria'
)




In [ ]:
# ================== TERNARY CLASSIFICATION ==================
# Repeat for ternary...
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Top 20 Bigrams Per Class")
print("="*80)
bigrams_ter = get_ngram_frequency_per_class(
    df_ter_eda,
    class_column='decisao_ternaria',
    n=2,
    top_n=20
)

plot_ngram_frequency_per_class(
    bigrams_ter,
    title=f"Top 20 Bigrams by Ternary Decision - {type_case_verbose} cases",
    n=2,
    class_column='decisao_ternaria'
)




In [ ]:
# ================== TRIGRAMS FREQUENCY ==================
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Top 20 Trigrams Per Class")
print("="*80)
trigrams_ter = get_ngram_frequency_per_class(
    df_ter_eda,
    class_column='decisao_ternaria',
    n=3,
    top_n=20
)

plot_ngram_frequency_per_class(
    trigrams_ter,
    title=f"Top 20 Trigrams by Ternary Decision - {type_case_verbose} cases",
    n=3,
    class_column='decisao_ternaria'
)



In [ ]:

# ================== TF-IDF BIGRAMS ==================
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Top 20 TF-IDF Bigrams Per Class")
print("="*80)
tfidf_bigrams_ter = get_tfidf_per_class(
    df_ter_eda,
    class_column='decisao_ternaria',
    ngram_range=(2, 2),
    top_n=20
)

plot_tfidf_per_class(
    tfidf_bigrams_ter,
    title=f"Top 20 TF-IDF Bigrams by Ternary Decision - {type_case_verbose} cases",
    n=2,
    class_column='decisao_ternaria'
)





In [ ]:

#================= TF-IDF TRIGRAMS ==================
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Top 20 TF-IDF Trigrams Per Class")
print("="*80)
tfidf_trigrams_ter = get_tfidf_per_class(
    df_ter_eda,
    class_column='decisao_ternaria',
    ngram_range=(3, 3),
    top_n=20
)

plot_tfidf_per_class(
    tfidf_trigrams_ter,
    title=f"Top 20 TF-IDF Trigrams by Ternary Decision - {type_case_verbose} cases",
    n=3,
    class_column='decisao_ternaria'
)





In [ ]:
# ================== 4GRAMS FREQUENCY ==================
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Top 20 4-grams Per Class")
print("="*80)
fourgrams_bin = get_ngram_frequency_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    n=4,
    top_n=20
)

plot_ngram_frequency_per_class(
    fourgrams_bin,
    title=f"Top 20 4-grams by Binary Decision - {type_case_verbose} cases",
    n=4,
    class_column='decisao_binaria'
)

In [ ]:
# ================== TF-IDF 4GRAMS ==================
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Top 20 TF-IDF 4-grams Per Class")
print("="*80)
tfidf_fourgrams_bin = get_tfidf_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    ngram_range=(4, 4),
    top_n=20
)

plot_tfidf_per_class(
    tfidf_fourgrams_bin,
    title=f"Top 20 TF-IDF 4-grams by Binary Decision - {type_case_verbose} cases",
    n=4,
    class_column='decisao_binaria'
)


In [ ]:
# ================== 4GRAMS FREQUENCY ==================
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Top 20 4-grams Per Class")
print("="*80)
fourgrams_ter = get_ngram_frequency_per_class(
    df_ter_eda,
    class_column='decisao_ternaria',
    n=4,
    top_n=20
)

plot_ngram_frequency_per_class(
    fourgrams_ter,
    title=f"Top 20 4-grams by Ternary Decision - {type_case_verbose} cases",
    n=4,
    class_column='decisao_ternaria'
)


In [ ]:
# ================== TF-IDF 4-GRAMS ==================
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Top 20 TF-IDF 4-grams Per Class")
print("="*80)
tfidf_fourgrams_ter = get_tfidf_per_class(
    df_ter_eda,
    class_column='decisao_ternaria',
    ngram_range=(4, 4),
    top_n=20
)

plot_tfidf_per_class(
    tfidf_fourgrams_ter,
    title=f"Top 20 TF-IDF 4-grams by Ternary Decision - {type_case_verbose} cases",
    n=4,
    class_column='decisao_ternaria'
)


In [ ]:
# ================== 5GRAMS FREQUENCY ==================
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Top 20 5-grams Per Class")
print("="*80)
fivegrams_bin = get_ngram_frequency_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    n=5,
    top_n=20
)

plot_ngram_frequency_per_class(
    fivegrams_bin,
    title=f"Top 20 5-grams by Binary Decision - {type_case_verbose} cases",
    n=5,
    class_column='decisao_binaria'
)

In [ ]:
# ================== TF-IDF 5GRAMS ==================
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Top 20 TF-IDF 5-grams Per Class")
print("="*80)
tfidf_fivegrams_bin = get_tfidf_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    ngram_range=(5, 5),
    top_n=20
)

plot_tfidf_per_class(
    tfidf_fivegrams_bin,
    title=f"Top 20 TF-IDF 5-grams by Binary Decision - {type_case_verbose} cases",
    n=5,
    class_column='decisao_binaria'
)

In [ ]:
# ================== 5GRAMS FREQUENCY ==================
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Top 20 5-grams Per Class")
print("="*80)
fivegrams_ter = get_ngram_frequency_per_class(
    df_ter_eda,
    class_column='decisao_ternaria',
    n=5,
    top_n=20
)

plot_ngram_frequency_per_class(
    fivegrams_ter,
    title=f"Top 20 5-grams by Ternary Decision - {type_case_verbose} cases",
    n=5,
    class_column='decisao_ternaria'
)

In [ ]:
# ================== TF-IDF 5GRAMS ==================
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Top 20 TF-IDF 5-grams Per Class")
print("="*80)
tfidf_fivegrams_ter = get_tfidf_per_class(
    df_ter_eda,
    class_column='decisao_ternaria',
    ngram_range=(5, 5),
    top_n=20
)

plot_tfidf_per_class(
    tfidf_fivegrams_ter,
    title=f"Top 20 TF-IDF 5-grams by Binary Decision - {type_case_verbose} cases",
    n=5,
    class_column='decisao_ternaria'
)

In [ ]:
from matplotlib import pyplot as plt
sb.barplot(data=df_bin_eda, x='decisao_binaria', y='text_length_raw', palette='bright')
title = f'Length of Text by Binary Decision - {type_case_verbose} cases'
plt.xlabel('Binary Decision')
plt.ylabel('Text Length (characters)')
plt.suptitle(title, fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(Path(f'./eda_viz/{type_case_folder}/basic_stats/text_length_by_binary_decision_{type_case}.png'), bbox_inches='tight',dpi=300)  # Save BEFORE show
plt.show()



In [ ]:
sb.barplot(data=df_ter_eda, x='decisao_ternaria', y='text_length_raw', palette='bright')
title = f'Length of Text by Ternary Decision - {type_case_verbose} cases'
plt.xlabel('Ternary Decision')
plt.ylabel('Text Length (characters)')
plt.xticks(rotation=45, ha='right')  # Rotate labels 45 degrees
plt.suptitle(title, fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(Path(f'./eda_viz/{type_case_folder}/basic_stats/text_length_by_ternary_decision_{type_case}.png'), bbox_inches='tight', dpi=300)  # Save BEFORE show
plt.show()



#### Removing stopwords

In [ ]:
IC_STOPWORDS_CONSERVATIVE = {
    # Only remove pure procedural/formalistic terms
    # Keep terms that might differentiate case outcomes
    
    # Procedural
    'tribunal', 'sentença', 'recurso', 'apelação', 'instância',
    'despacho', 'acórdão', 'processo',
    
    # Actors
    'ré', 'réu', 'autora', 'autor', 'recorrente', 'recorrido',
    'parte', 'partes',
    
    # Evidence
    'factos provados', 'matéria facto', 'prova', 'provado',
    
    # References
    'art', 'artigo', 'nº', 'cc', 'cpc', 'código',
    'artigo código', 'art cc', 'art nº',
    
    # Generic connectors
    'tal', 'assim', 'sendo', 'tendo', 'facto', 'factos',
    'caso', 'termos', 'qualquer', 'outro',
    
    # Decision markers
    'decisão', 'decidir', 'julgar',
}


#################################

IC_STOPWORDS = {
    # ====================
    # PROCEDURAL TERMS (Always remove - pure legal formalism)
    # ====================
    'tribunal', 'tribunal quo', 'sentença recorrida', 'petição inicial',
    'base instrutória', 'fls autos', 'autos', 'processo', 'recurso',
    'apelação', 'relação', 'supremo', 'stj', 'trp', 'trl', 'trc', 'tre', 'trg',
    'instância', 'primeira instância', 'segunda instância',
    'despacho', 'acórdão', 'decisão recorrida',
    
    # ====================
    # GENERIC LEGAL ACTORS (Remove - appear in ALL cases)
    # ====================
    'ré', 'réu', 'autora', 'autor', 'demandante', 'demandada', 'demandado',
    'recorrente', 'recorrido', 'recorrida', 'apelante', 'apelado', 'apelada',
    'parte', 'partes', 'contraente', 'contraentes',
    
    # ====================
    # EVIDENCE & PROCEDURE (Remove - pure formalism)
    # ====================
    'factos provados', 'matéria facto', 'prova', 'provado', 'provados',
    'alegação', 'alegações', 'alegado', 'alegada',
    'testemunha', 'testemunhas', 'depoimento', 'documento', 'documentos',
    'articulado', 'contestação',
    
    # ====================
    # LEGAL REFERENCES (Remove - citations, not content)
    # ====================
    'art', 'artigo', 'artº', 'nº', 'cc', 'cpc', 'código civil', 
    'código processo', 'processo civil', 'artigo código',
    'art cc', 'art nº', 'art código',
    'regime', 'lei', 'decreto', 'dl',
    
    # ====================
    # GENERIC LEGAL CONNECTORS (Remove - no semantic value)
    # ====================
    'tal', 'assim', 'sendo', 'tendo', 'tendo sido', 'ter sido',
    'facto', 'factos', 'caso', 'termos', 'termo',
    'qualquer', 'outro', 'outra', 'outros', 'outras',
    'modo', 'forma', 'ainda', 'só', 'apenas',
    
    # ====================
    # DECISION TERMS (Remove - outcome markers, not reasoning)
    # ====================
    # 'decisão', 'decidir', 'julgar', 'julga', 'julgado', 'julgada',
    # 'procedente', 'improcedente', 'parcialmente procedente',
    # 'provido', 'negado provimento', 'provimento',
    # 'confirmada', 'revogada', 'alterada', 'mantida',
    
    # ====================
    # MAYBE REMOVE - HIGHLY DOMAIN-SPECIFIC (Review these carefully!)
    # ====================
    # These appear in top terms but might carry semantic meaning
    # Keep for now, review after seeing results. Let's stake out now
    'contrato', 'promessa', 'compra', 'venda', 'pagamento', 
    'valor', 'incumprimento', 'resolução',
}

In [ ]:
DV_STOPWORDS = {
    # ====================
    # PROCEDURAL TERMS (Always remove - pure legal formalism)
    # ====================
    'tribunal', 'tribunal quo', 'sentença recorrida', 'petição inicial',
    'base instrutória', 'fls autos', 'autos', 'processo', 'recurso',
    'apelação', 'relação', 'supremo', 'stj', 'trp', 'trl', 'trc', 'tre', 'trg',
    'instância', 'primeira instância', 'segunda instância',
    'despacho', 'acórdão', 'decisão recorrida',
    
    # ====================
    # GENERIC LEGAL ACTORS (Remove - appear in ALL cases)
    # ====================
    'ré', 'réu', 'autora', 'autor', 'demandante', 'demandada', 'demandado',
    'recorrente', 'recorrido', 'recorrida', 'apelante', 'apelado', 'apelada',
    'parte', 'partes',
    
    # ====================
    # EVIDENCE & PROCEDURE (Remove - pure formalism)
    # ====================
    'factos provados', 'matéria facto', 'prova', 'provado', 'provados',
    'alegação', 'alegações', 'alegado', 'alegada',
    'testemunha', 'testemunhas', 'depoimento', 'documento', 'documentos',
    'articulado', 'contestação',
    
    # ====================
    # LEGAL REFERENCES (Remove - citations, not content)
    # ====================
    'art', 'artigo', 'artº', 'nº', 'código penal', 'código processo', 'código',
    'processo penal', 'artigo código', 'art nº', 'lei','artigo'
    
    # ====================
    # GENERIC LEGAL CONNECTORS (Remove - no semantic value)
    # ====================
    'tal', 'assim', 'sendo', 'tendo', 'tendo sido', 'ter sido',
    'facto', 'factos', 'caso', 'termos', 'termo',
    'qualquer', 'outro', 'outra', 'outros', 'outras',
    'modo', 'forma', 'ainda', 'só', 'apenas',
    
    # ====================
    # GENERIC CRIME TERMS (Keep specific ones, remove generic)
    # ====================
    'crime', 'arguido', 'arguida',  # Too generic in criminal cases
    
    # ====================
    # KEEP THESE - SEMANTIC VALUE FOR DV CASES:
    # ====================
    # ❌ NÃO REMOVER:
    # - 'ofendida', 'ofendido' (vítima específica em DV)
    # - 'violência', 'violência doméstica' (categoria central)
    # - 'agressão', 'agressões', 'agressivo'
    # - 'ameaça', 'ameaças', 'ameaçou'
    # - 'ofensa', 'ofensas', 'integridade física'
    # - 'injúria', 'injúrias', 'difamação'
    # - 'coação', 'coacção'
    # - 'perseguição', 'perseguir', 'stalking'
    # - 'habitação', 'residência', 'domicílio' (contexto doméstico)
    # - 'relação', 'cônjuge', 'companheiro', 'ex-cônjuge', 'ex-companheiro'
    # - 'filho', 'filha', 'filhos', 'menor', 'menores' (vítimas vulneráveis)
    # - 'arma', 'armas', 'faca', 'arma branca'
    # - 'álcool', 'alcoolizado', 'embriaguez', 'bebida'
    # - 'ciúme', 'ciúmes', 'ciumenta', 'ciumento'
    # - 'separação', 'divórcio', 'rutura'
    # - 'dolo', 'dolo direto', 'dolo eventual', 'intenção'
    # - 'reincidência', 'reincidente', 'antecedentes'
    # - 'medida coação', 'prisão preventiva', 'pulseira electrónica'
    # - 'pena', 'prisão', 'suspensão pena', 'pena suspensa'
    # - 'gravidade', 'intensidade', 'duração'
    # - 'sofrimento', 'dor', 'trauma', 'psicológico'
}

# Basically same as IC cases, but removing some terms that are too generic in DV cases


In [ ]:

def remove_legal_stopwords(tokens, stopword_set='conservative'):
    """
    Remove legal stopwords from token list.
    
    Args:
        tokens: List of tokens
        stopword_set: 'conservative', 'standard', or 'aggressive'
    
    Returns:
        List of tokens with legal stopwords removed
    """
    if stopword_set == 'conservative':
        legal_stops = IC_STOPWORDS_CONSERVATIVE
    # elif stopword_set == 'aggressive':
    #     legal_stops = IC_STOPWORDS_AGGRESSIVE
    else:  # standard
        legal_stops = DV_STOPWORDS
    
    if isinstance(tokens, list):
        return [token for token in tokens if token not in legal_stops]
    return tokens




In [ ]:
# # # Try conservative first
# df_bin_eda['tokens_no_legal_stopwords_conservative'] = df_bin_eda['tokens_no_stopwords'].apply(
#     lambda x: remove_legal_stopwords(x, stopword_set='conservative')
# )

df_bin_eda['tokens_no_legal_stopwords_standard'] = df_bin_eda['tokens_no_stopwords'].apply(
    lambda x:  remove_legal_stopwords(x, stopword_set='standard')
)

In [ ]:
# resultados_conservative = get_most_common_words_per_class(
#     df_bin_eda,
#     class_column='decisao_binaria',
#     text_column='tokens_no_legal_stopwords_conservative',  # Conservative version
#     top_n=20
# )

resultados_standard = get_most_common_words_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    text_column='tokens_no_legal_stopwords_standard',  # Standard version
    top_n=20
)

# plot_most_common_words_per_class(resultados_conservative,
#                                  title=f"Top 20 Most Common Words by Binary Decision (Legal Stopwords Removed - Conservative) - {type_case_verbose} cases",
#     class_column='decisao_binaria'
    
# )

plot_most_common_words_per_class(resultados_standard,
                                 title=f"Top 20 Most Common Words by Binary Decision (Legal Stopwords Removed - Standard) - {type_case_verbose} cases",
    class_column='decisao_binaria'
    
)

In [ ]:
# resultados_tfidf_conservative = get_distinctive_words_per_class(
#     df_bin_eda,
#     class_column='decisao_binaria',
#     text_column='tokens_no_legal_stopwords_conservative',  # Conservative version
#     top_n=20
    
    
# )

resultados_tfidf_standard = get_distinctive_words_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    text_column='tokens_no_legal_stopwords_standard',  # Standard version
    top_n=20
)

# plot_most_tf_idf_per_class(resultados_tfidf_conservative,
#     title=f"Top 20 TF-IDF Words by Binary Decision (Legal Stopwords Removed- Conservative) - {type_case_verbose} cases",
#     class_column='decisao_binaria'
# )

plot_most_tf_idf_per_class(resultados_tfidf_standard,
    title=f"Top 20 TF-IDF Words by Binary Decision (Legal Stopwords Removed- Standard) - {type_case_verbose} cases",
    class_column='decisao_binaria'
)


    

### Interesting Stats and ViZ
- Descriptors frequency (outside the main ones, e.g 'VIOLÊNCIA DOMÉSTICA', 'INCUMPRIMENTO DO/DE CONTRATO/CONTRATUAL).
- Trafnorming judge names to gender (male, female, non descriptive(meaning we cant extract gender by the names, for example can be just surnames)). Then get class distribution based on gender
- Class distribution based on Tribunal

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from collections import Counter
from pathlib import Path
import ast

def analyze_descriptors_per_class(df, class_column, descriptor_column='descritores', 
                                   exclude_terms=None, top_n=15):
    """
    Analyze frequency of legal descriptors per class.
    
    Args:
        df: DataFrame with legal cases
        class_column: Name of column with class labels ('decisao_binaria' or 'decisao_ternaria')
        descriptor_column: Name of column with descriptor lists (default: 'descritores')
        exclude_terms: List of generic terms to exclude (e.g., ['INCUMPRIMENTO', 'VIOLÊNCIA DOMÉSTICA'])
        top_n: Number of top descriptors to show per class
    
    Returns:
        dict: Dictionary with class labels as keys and descriptor frequency dicts as values
    """
    
    if exclude_terms is None:
        exclude_terms = []
    
    # Normalize exclude terms (lowercase for matching)
    exclude_terms_lower = [term.lower() for term in exclude_terms]
    
    results = {}
    classes = df[class_column].unique()
    
    print(f"\n{'='*80}")
    print(f"DESCRIPTOR FREQUENCY ANALYSIS - Top {top_n} per class")
    print(f"{'='*80}\n")
    
    for class_label in classes:
        # Filter by class
        df_class = df[df[class_column] == class_label]
        
        # Collect all descriptors from this class
        all_descriptors = []
        for desc_list in df_class[descriptor_column]:
            # desc_list = list(desc_list)  # Wrong, this transforms string into list of chars
            desc_list = ast.literal_eval(desc_list)
            
            if isinstance(desc_list, list):
                # print(f'Descritors is a list: {desc_list}')
                # Normalize and filter
                for desc in desc_list:
                    desc_clean = desc.strip().upper()
                    desc_lower = desc_clean.lower()
                    
                    # Skip generic terms
                    skip = False
                    for exclude in exclude_terms_lower:
                        if exclude in desc_lower:
                            skip = True
                            break
                    # print(f"Descriptor: {desc_clean}, Skip: {skip}")
                    if not skip and len(desc_clean) > 3:  # Skip very short descriptors
                        all_descriptors.append(desc_clean)
            else:
                print(f'Descritors is NOT a list: {desc_list}')
        # print(f'All descriptors: {all_descriptors}')
        # break       
        # Count frequencies
        descriptor_counts = Counter(all_descriptors)
        top_descriptors = descriptor_counts.most_common(top_n)
        # print(f'top descriptors: {top_descriptors}')
        
        results[class_label] = dict(top_descriptors)
        
        # Print results
        print(f"{'='*60}")
        print(f"Class: {class_label} ({len(df_class)} cases)")
        print(f"{'='*60}")
        print(f"Total unique descriptors: {len(descriptor_counts)}")
        print(f"\nTop {top_n} descriptors:")
        for desc, count in top_descriptors:
            percentage = (count / len(df_class)) * 100
            print(f"  {desc:50s} - {count:4d} ({percentage:5.1f}%)")
        print()
    
    return results

# break

def plot_descriptors_per_class(results, class_column, title="Top Legal Descriptors by Class"):
    """
    Create horizontal bar plots for descriptor frequencies per class.
    
    Args:
        results: Dictionary from analyze_descriptors_per_class()
        class_column: Name of class column (for file naming)
        title: Title for the plot
    """
    num_classes = len(results)
    fig, axes = plt.subplots(1, num_classes, figsize=(12*num_classes, 10))
    
    # Handle single class case
    if num_classes == 1:
        axes = [axes]
    
    for idx, (class_label, descriptor_dict) in enumerate(results.items()):
        descriptors = list(descriptor_dict.keys())
        counts = list(descriptor_dict.values())
        
        # Truncate long descriptor names for readability
        descriptors_short = [desc[:40] + '...' if len(desc) > 40 else desc 
                            for desc in descriptors]
        
        sb.barplot(x=counts, y=descriptors_short, ax=axes[idx], palette='viridis')
        axes[idx].set_title(f'Class: {class_label}', fontsize=14, fontweight='bold')
        axes[idx].set_xlabel('Frequency', fontsize=12)
        axes[idx].set_ylabel('Legal Descriptor', fontsize=12)
        axes[idx].tick_params(axis='y', labelsize=9)
    
    plt.suptitle(title, fontsize=16, y=0.98, fontweight='bold')
    plt.tight_layout()
    
    # Save plot
    plt.savefig(
        Path(f'./eda_viz/{type_case_folder}/descriptors/descriptors_frequency_{type_case}_{class_column}.png'),
        bbox_inches='tight', 
        dpi=300
    )
    
    plt.show()




In [ ]:
# ============================================================
# USAGE EXAMPLE
# ============================================================

# Define generic terms to exclude (case-specific)
if type_case == 'ic':
    exclude_generic = [
        'INCUMPRIMENTO', 
        'CONTRATO',
        'PROCESSO',  # Too generic,
        'Nº DO DOCUMENTO',
        'DATA DO ACORDÃO',
        'RESOLUÇÃO' # significa quebrar o contrato por incumprimento logo nao adiciona valor
    ]
elif type_case == 'dv':
    exclude_generic = [
        'VIOLÊNCIA DOMÉSTICA',
        'VIOLÊNCIA',
        'PROCESSO',
        'Nº DO DOCUMENTO',
        'DATA DO ACORDÃO',
    ]

# ================== BINARY CLASSIFICATION ==================
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Descriptor Analysis")
print("="*80)

descriptors_bin = analyze_descriptors_per_class(
    df_bin_eda,
    class_column='decisao_binaria',
    exclude_terms=exclude_generic,
    top_n=15
)

plot_descriptors_per_class(
    descriptors_bin,
    class_column='decisao_binaria',
    title=f"Top 15 Legal Descriptors by Binary Decision - {type_case_verbose} cases"
)


# ================== TERNARY CLASSIFICATION ==================
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Descriptor Analysis")
print("="*80)

descriptors_ter = analyze_descriptors_per_class(
    df_ter_eda,
    class_column='decisao_ternaria',
    exclude_terms=exclude_generic,
    top_n=15
)

plot_descriptors_per_class(
    descriptors_ter,
    class_column='decisao_ternaria',
    title=f"Top 15 Legal Descriptors by Ternary Decision - {type_case_verbose} cases"
)

Casos IC:
- Mora :  No contexto do não cumprimento das obrigações, mas ainda num sentido genérico, a «mora» designa todos os casos de não cumprimento no momento desejável, em que se mantém a obrigação, embora a cumprir num momento posterior. (...) , incumprimento definiti-vo e impossibilidade imputável —, de situações de não cumprimento imputáveis ao devedor, ou seja, que são devidas a culpa sua ou a «culpa» dos seus auxiliares ou representantes legais. O que é específico da mora é que a obrigação se mantém, apesar do problema existente. Assim, o credor mantém o direito de exigir o próprio cumpri-mento da obrigação, inclusive por via judicial, e não apenas uma indemnização. E o devedor mantém o «direito» de cumprir, ou seja, mantém uma série de direitos associados ao cumprimento no caso de vir efectivamente a cumprir (por exemplo, o direito a ser pago por isso, se for o caso).  [Fonte - Lexionário Diário da República](https://diariodarepublica.pt/dr/lexionario/termo/mora) 


- INTERPELAÇÃO ADMONITÓRIA: A interpelação admonitória consiste no ato pelo qual o credor demanda o devedor em mora para que este realize a obrigação a que está vinculado, sob pena de entrar em incumprimento definitivo. Na interpelação admonitória, o credor deve fixar um prazo suplementar razoável ao devedor para que cumpra, sob pena de, não o fazendo, se considerar a obrigação definitivamente incumprida. Passado o prazo admonitório, dá-se o incumprimento definitivo, tendo o credor todos os direitos provenientes do incumprimento definitivo, designadamente o direito de resolver o contrato.


---


Casos DV:
- DECLARAÇÕES PARA MEMÓRIA FUTURA: são depoimentos de uma testemunha ou vítima que são recolhidos previamente a um julgamento para preservar a prova, quando se prevê que essa pessoa poderá não estar disponível ou será prejudicada ao ser ouvida em tribunal.  Usado em  caso de doença grave ou de deslocação para o estrangeiro de uma testemunha, que previsivelmente a impeça de ser ouvida em julgamento, bem como nos casos de vítima de crime de tráfico de órgãos humanos, tráfico de pessoas ou contra a liberdade e autodeterminação sexual..


- BEM JURIDICO PROTEGIDO: Proteção dos direitos fundamentais das vítimas (diretas e indiretas), como saúde fisica e mental, liberdade, dignidade humana, etc. Isto torna a violência domestica mais grave e mais complexa do que casos de violencia sem esta relação, pois vitimas e agressores tem relação de proximidade e por vezes até de dependencia, podendo causar danos alem dos convencionais, particularmente psicologicos e sociais. [FONTE - NOTA PRÁTICA MINISTÉRIO PÚBLICO ](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://gfcjivd.ministeriopublico.pt/sites/default/files/documentos/pdf/nota-pratica-1-2023-viol-domestica-bem-juridico_0.pdf)

- HABEAS CORPUS: instrumento jurídico para proteger a liberdade de locomoção de um indivíduo contra prisões ilegais ou abuso de poder. Basicamente permite retificar ou impedir detenções ou prisões consideradas ilegais, quer em fundamento como em questões prazos e conformidades. [CPP- Art 220 - Lei 78/87](https://diariodarepublica.pt/dr/legislacao-consolidada/decreto-lei/1987-34570075-50535475)

###### Minha interpetação:

Okay analisei os descriptores e a conclusão que cheguei é que eles não tem grande valor preditivo de qualquer uma das classes, sendo mais sinais comuns relativo a casos de incumprimento de contrato, como mora (demora no cumprimento de requisitos do contrato), abuso de direito (incumprimento da lei para um beneficio econocimo-social), etc. para violencia domestica tambem nao e possivel detetar grandes padroões preditivos de decisao, contudo mostram as complexidades do tipo de caso com procedimento de impedimento de prisao como habeas corpus, proteção das vitimas com bem juridico protegido e declarações futuras e outros crimes graves associados como violações e homicidios qualificados.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from pathlib import Path

def analyze_decisions_by_tribunal(df, class_column, tribunal_column='tribunal'):
    """
    Analyze decision distribution by tribunal (court).
    Groups all 'JP_*' tribunals into a single 'Julgados de Paz' category.
    
    Args:
        df: DataFrame with legal cases
        class_column: Name of column with class labels ('decisao_binaria' or 'decisao_ternaria')
        tribunal_column: Name of column with tribunal names (default: 'tribunal')
    
    Returns:
        pd.DataFrame: Cross-tabulation of decisions by tribunal with percentages
    """
    
    # Create a copy to avoid modifying original data
    df_analysis = df.copy()
    
    # Group all JP_* tribunals into 'Julgados de Paz'
    df_analysis[tribunal_column] = df_analysis[tribunal_column].apply(
        lambda x: 'JP' if str(x).startswith('JP') else x
    )
    
    # Create cross-tabulation
    ct = pd.crosstab(
        df_analysis[tribunal_column], 
        df_analysis[class_column], 
        margins=True,
        margins_name='Total'
    )
    
    # Calculate percentages (row-wise)
    ct_pct = pd.crosstab(
        df_analysis[tribunal_column], 
        df_analysis[class_column], 
        normalize='index'
    ) * 100
    
    print(f"\n{'='*80}")
    print(f"DECISION DISTRIBUTION BY TRIBUNAL - {class_column.upper()}")
    print(f"{'='*80}\n")
    
    print("Absolute Counts:")
    print(ct)
    print("\n" + "="*80 + "\n")
    
    print("Percentages (by row):")
    print(ct_pct.round(2))
    print("\n" + "="*80 + "\n")
    
    return ct, ct_pct


def plot_decisions_by_tribunal(df, class_column, tribunal_column='tribunal', 
                                title="Decision Distribution by Tribunal"):
    """
    Create stacked bar plot showing decision distribution by tribunal.
    Groups all 'JP_*' tribunals into 'Julgados de Paz' before plotting.
    Adds percentage labels inside each bar segment.
    
    Args:
        df: DataFrame with legal cases
        class_column: Name of column with class labels
        tribunal_column: Name of column with tribunal names
        title: Title for the plot
    """
    
    # Create a copy and group JP tribunals
    df_plot = df.copy()
    df_plot[tribunal_column] = df_plot[tribunal_column].apply(
        lambda x: 'JP' if str(x).startswith('JP') else x
    )
    
    # Calculate percentages for plotting
    ct_pct = pd.crosstab(
        df_plot[tribunal_column], 
        df_plot[class_column], 
        normalize='index'
    ) * 100
    
    # Get tribunal counts (for labeling)
    tribunal_counts = df_plot[tribunal_column].value_counts()
    
    # Sort by total cases (descending)
    ct_pct = ct_pct.loc[tribunal_counts.index]
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Stacked bar plot
    ct_pct.plot(
        kind='barh', 
        stacked=True, 
        ax=ax,
        colormap='viridis',
        width=0.8
    )
    
    # Add percentage labels inside bars
    for i, (tribunal, row) in enumerate(ct_pct.iterrows()):
        cumulative = 0
        for j, (decision, value) in enumerate(row.items()):
            # Only add label if percentage is significant (>3%)
            if value > 3:
                # Position label at center of bar segment
                x_pos = cumulative + value / 2
                # Add white text with red outline for better visibility
                ax.text(
                    x_pos, i, f'{value:.1f}%',
                    ha='center', va='center',
                    fontsize=10, fontweight='bold',
                    color='white',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='red', 
                             edgecolor='darkred', alpha=0.8)
                )
            cumulative += value
    
    # Customize
    ax.set_xlabel('Percentage (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Tribunal', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.legend(title='Decision', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(axis='x', alpha=0.3)
    
    # Add case counts to y-axis labels
    labels = [f"{trib} (n={tribunal_counts[trib]})" for trib in ct_pct.index]
    ax.set_yticklabels(labels, fontsize=10)
    
    plt.tight_layout()
    
    # Save plot
    plt.savefig(
        Path(f'./eda_viz/{type_case_folder}/tribunal/decisions_by_tribunal_{type_case}_{class_column}.png'),
        bbox_inches='tight', 
        dpi=300
    )
    
    plt.show()




In [ ]:
# ============================================================
# USAGE
# ============================================================

# ================== BINARY CLASSIFICATION ==================
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Tribunal Analysis")
print("="*80)

ct_bin, ct_pct_bin = analyze_decisions_by_tribunal(
    df_bin_eda,
    class_column='decisao_binaria'
)

plot_decisions_by_tribunal(
    df_bin_eda,
    class_column='decisao_binaria',
    title=f"Binary Decision Distribution by Tribunal - {type_case_verbose} cases"
)


# ================== TERNARY CLASSIFICATION ==================
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Tribunal Analysis")
print("="*80)

ct_ter, ct_pct_ter = analyze_decisions_by_tribunal(
    df_ter_eda,
    class_column='decisao_ternaria'
)

plot_decisions_by_tribunal(
    df_ter_eda,
    class_column='decisao_ternaria',
    title=f"Ternary Decision Distribution by Tribunal - {type_case_verbose} cases"
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from pathlib import Path
# from gender_guesser_br import Genero
import gender_guesser.detector as gender

def extract_first_name(full_name):
    """
    Extract first name from full judge name.
    Handles various formats and titles.
    
    Args:
        full_name: Full name string (e.g., "Dr. João Silva")
    
    Returns:
        str: First name only, or None if can't extract
    """
    if pd.isna(full_name) or not isinstance(full_name, str):
        return None
    
    # Remove common titles
    name = full_name.strip()
    # titles = ['Dr.', 'Dra.', 'Des.', 'Juiz', 'Juíza', 'Desembargador', 'Desembargadora']
    # for title in titles:
    #     name = name.replace(title, '').strip()
    
    # Split by spaces and get first name
    parts = name.split()
    if parts:
        first_name = parts[0].strip()
        # Remove any remaining punctuation
        first_name = first_name.replace('.', '').replace(',', '')
        return first_name.lower()  # Lowercase for gender-guesser
    
    return None


def classify_judge_gender(df, judge_column='juiz_relator'):
    """
    Classify judge gender based on first name using gender-guesser.
    
    Args:
        df: DataFrame with legal cases
        judge_column: Name of column with judge names (default: 'juiz_relator')
    
    Returns:
        pd.DataFrame: DataFrame with added 'judge_gender' column
    """
    
    # Create copy to avoid modifying original
    df_gender = df.copy()
    
    # Extract first names
    df_gender['judge_first_name'] = df_gender[judge_column].apply(extract_first_name)
    
    # Initialize gender detector
    detector = gender.Detector(case_sensitive=False)
    
    # Classify gender
    gender_results = []
    for first_name in df_gender['judge_first_name']:
        if first_name:
            try:
                result = detector.get_gender(first_name)
                gender_results.append(result)
            except Exception as e:
                print(f"Error classifying name '{first_name}': {e}")
                gender_results.append('unknown')
        else:
            gender_results.append('unknown')
    
    df_gender['judge_gender_raw'] = gender_results
    
    # Simplify gender categories for analysis
    def simplify_gender(gender_raw):
        """
        Simplify gender categories:
        - male, mostly_male -> Masculino
        - female, mostly_female -> Feminino
        - andy, unknown -> Indeterminado
        """
        if gender_raw in ['male', 'mostly_male']:
            return 'Masculino'
        elif gender_raw in ['female', 'mostly_female']:
            return 'Feminino'
        else:
            return 'Indeterminado'
    
    df_gender['judge_gender'] = df_gender['judge_gender_raw'].apply(simplify_gender)
    
    # Print summary statistics
    print(f"\n{'='*80}")
    print(f"JUDGE GENDER CLASSIFICATION SUMMARY")
    print(f"{'='*80}\n")
    
    print("Raw Gender Distribution:")
    print(df_gender['judge_gender_raw'].value_counts())
    print(f"\nSimplified Gender Distribution:")
    print(df_gender['judge_gender'].value_counts())
    print(f"\nTotal cases: {len(df_gender)}")
    print(f"Successfully classified: {(df_gender['judge_gender'] != 'Indeterminado').sum()}")
    print(f"Classification rate: {((df_gender['judge_gender'] != 'Indeterminado').sum() / len(df_gender)) * 100:.1f}%")
    
    return df_gender


def analyze_decisions_by_judge_gender(df, class_column, gender_column='judge_gender'):
    """
    Analyze decision distribution by judge gender.
    
    Args:
        df: DataFrame with legal cases and judge gender
        class_column: Name of column with class labels ('decisao_binaria' or 'decisao_ternaria')
        gender_column: Name of column with judge gender (default: 'judge_gender')
    
    Returns:
        tuple: (ct, ct_pct) - Cross-tabulation tables
    """
    
    # Create cross-tabulation
    ct = pd.crosstab(
        df[gender_column], 
        df[class_column], 
        margins=True,
        margins_name='Total'
    )
    
    # Calculate percentages (row-wise)
    ct_pct = pd.crosstab(
        df[gender_column], 
        df[class_column], 
        normalize='index'
    ) * 100
    
    print(f"\n{'='*80}")
    print(f"DECISION DISTRIBUTION BY JUDGE GENDER - {class_column.upper()}")
    print(f"{'='*80}\n")
    
    print("Absolute Counts:")
    print(ct)
    print("\n" + "="*80 + "\n")
    
    print("Percentages (by row):")
    print(ct_pct.round(2))
    print("\n" + "="*80 + "\n")
    
    return ct, ct_pct


def plot_decisions_by_judge_gender(df, class_column, gender_column='judge_gender',
                                    title="Decision Distribution by Judge Gender"):
    """
    Create stacked bar plot showing decision distribution by judge gender.
    
    Args:
        df: DataFrame with legal cases and judge gender
        class_column: Name of column with class labels
        gender_column: Name of column with judge gender
        title: Title for the plot
    """
    
    # Calculate percentages for plotting
    ct_pct = pd.crosstab(
        df[gender_column], 
        df[class_column], 
        normalize='index'
    ) * 100
    
    # Get gender counts (for labeling)
    gender_counts = df[gender_column].value_counts()
    
    # Sort by total cases (descending)
    ct_pct = ct_pct.loc[gender_counts.index]
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Stacked bar plot
    ct_pct.plot(
        kind='barh', 
        stacked=True, 
        ax=ax,
        colormap='viridis',
        width=0.7
    )
    
    
    #Interessante ser assim que adiciona labels de percentagem dentro das barras
    for i,(gender,row) in enumerate(ct_pct.iterrows()):
        cumulative = 0
        for j,(decision,value) in enumerate(row.items()):
            # Adding percentage labels inside bars if significant (>3%)
            if value > 3:
                x_pos = cumulative + value / 2
                ax.text(
                    x_pos,i,f'{value:.1f}%',
                    ha='center', va='center',
                    fontsize=10, fontweight='bold',
                    color='white',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='red', 
                             edgecolor='darkred', alpha=0.8)
                    
                )
                
                cumulative += value
    
    # Customize
    ax.set_xlabel('Percentage (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Judge Gender', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.legend(title='Decision', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(axis='x', alpha=0.3)
    
    # Add case counts to y-axis labels
    labels = [f"{gender} (n={gender_counts[gender]})" for gender in ct_pct.index]
    ax.set_yticklabels(labels, fontsize=11)
    
    plt.tight_layout()
    
    # Save plot
    plt.savefig(
        Path(f'./eda_viz/{type_case_folder}/judge_gender/decisions_by_judge_gender_{type_case}_{class_column}.png'),
        bbox_inches='tight', 
        dpi=300
    )
    
    plt.show()




In [ ]:
# ============================================================
# USAGE
# ============================================================

# First, classify judge gender (only need to do once per dataframe)
print("\n" + "="*80)
print("CLASSIFYING JUDGE GENDER")
print("="*80)

df_bin_eda_gender = classify_judge_gender(df_bin_eda)
df_ter_eda_gender = classify_judge_gender(df_ter_eda)


# ================== BINARY CLASSIFICATION ==================
print("\n" + "="*80)
print("BINARY CLASSIFICATION - Judge Gender Analysis")
print("="*80)

ct_gender_bin, ct_pct_gender_bin = analyze_decisions_by_judge_gender(
    df_bin_eda_gender,
    class_column='decisao_binaria'
)

plot_decisions_by_judge_gender(
    df_bin_eda_gender,
    class_column='decisao_binaria',
    title=f"Binary Decision Distribution by Judge Gender - {type_case_verbose} cases"
)


# ================== TERNARY CLASSIFICATION ==================
print("\n" + "="*80)
print("TERNARY CLASSIFICATION - Judge Gender Analysis")
print("="*80)

ct_gender_ter, ct_pct_gender_ter = analyze_decisions_by_judge_gender(
    df_ter_eda_gender,
    class_column='decisao_ternaria'
)

plot_decisions_by_judge_gender(
    df_ter_eda_gender,
    class_column='decisao_ternaria',
    title=f"Ternary Decision Distribution by Judge Gender - {type_case_verbose} cases"
)

In [ ]:
df_bin_eda_gender[['judge_first_name','judge_gender']].head(100)

### Readability and Emotional Tone

- sentiment analysis (to understand if domestic violence cases are in fact more emotional than contract breaches or if there is no signigicant difference)
- flesch_reading_ease para comparar a dificuldade de leitura entre casos de violencia domestica vs contract breach, perceber se a diferença é signigicativa ou não.


BERT model:
-  from https://github.com/RenanPeres/RenanPeres/tree/main/thesis and https://huggingface.co/renanperes/BERPT
- https://huggingface.co/pysentimiento/bertweet-pt-sentiment/blob/main/README.md

Lexicon Model: 
- SentiLex-PT: https://b2find.eudat.eu/dataset/b6bd16c2-a8ab-598f-be41-1e7aeecd60d3

In [ ]:
#Import both cases dfs 
import pandas as pd
from pathlib import Path 


df_ic_bin_eda = pd.read_csv(Path('./../../data/processed_data/eda/binary/df_acordaos_ic_eda_binary.csv'))
df_dv_bin_eda = pd.read_csv(Path('./../../data/processed_data/eda/binary/df_acordaos_dv_eda_binary.csv'))

In [ ]:
print(f'{len(df_ic_bin_eda)} cases for Contract Breach EDA')
print(f'{len(df_dv_bin_eda)} cases for Domestic Violence EDA')

In [ ]:
df_ic_bin_eda['clean_text'] = df_ic_bin_eda['texto_integral_sem_decisao'].apply(clean_text)
df_dv_bin_eda['clean_text'] = df_dv_bin_eda['texto_integral_sem_decisao'].apply(clean_text)


In [ ]:
from pathlib import Path

file_lema = Path('./../../SentiLex-PT02/SentiLex-lem-PT02.txt')
file_flex = Path('./../../SentiLex-PT02/SentiLex-flex-PT02.txt')
with open(
    file_lema,
    'r', encoding='utf-8'
) as f:
    senti_lex_lema_lines = f.readlines()

with open(
    file_flex,
    'r', encoding='utf-8'
) as f:
    senti_lex_flex_lines = f.readlines()     
        
print(len(senti_lex_lema_lines))
print(len(senti_lex_flex_lines))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
from pathlib import Path
from scipy import stats
import re
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# SENTILEX-PT LOADER
# ============================================================

def load_sentilex_flex(filepath=file_flex):
    """
    Load SentiLex-PT inflected forms lexicon.
    
    Format per line:
    inflected_form,lemma.PoS=X;TG=Y;POL:N0=Z;ANOT=W
    
    Returns:
        dict: {word: polarity} where polarity is -1, 0, or 1
    """
    
    sentiment_dict = {}
    
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                
                try:
                    # Split on first comma: inflected_form,attributes
                    parts = line.split(',', 1)
                    if len(parts) != 2:
                        continue
                    
                    inflected_form = parts[0].strip()
                    attributes = parts[1]
                    
                    # Extract polarity using regex: POL:N0=X
                    pol_match = re.search(r'POL:N0=(-?\d+)', attributes)
                    if pol_match:
                        polarity = int(pol_match.group(1))
                        
                        # Handle multi-word idioms (spaces in inflected_form)
                        if ' ' in inflected_form:
                            # Store each word separately AND as phrase
                            sentiment_dict[inflected_form] = polarity
                            # Also store individual words (for partial matches)
                            for word in inflected_form.split():
                                if word not in sentiment_dict:
                                    sentiment_dict[word] = polarity
                        else:
                            # Single word
                            sentiment_dict[inflected_form] = polarity
                
                except Exception as e:
                    print(f"⚠️  Error parsing line {line_num}: {e}")
                    continue
        
        print(f"✓ Loaded SentiLex-PT: {len(sentiment_dict)} entries")
        print(f"   - Positive words: {sum(1 for v in sentiment_dict.values() if v > 0)}")
        print(f"   - Negative words: {sum(1 for v in sentiment_dict.values() if v < 0)}")
        print(f"   - Neutral words: {sum(1 for v in sentiment_dict.values() if v == 0)}")
        
        return sentiment_dict
    
    except FileNotFoundError:
        print(f"❌ SentiLex file not found at: {filepath}")
        print(f"   Please download from: https://b2find.eudat.eu/dataset/b6bd16c2-a8ab-598f-be41-1e7aeecd60d3")
        return None


# ============================================================
# SENTIMENT CALCULATION
# ============================================================

def calculate_sentiment_sentilex(text, sentiment_dict):
    """
    Calculate sentiment score using SentiLex-PT lexicon.
    
    Args:
        text: String of text (already cleaned/tokenized)
        sentiment_dict: Dictionary from load_sentilex_flex()
    
    Returns:
        dict: {
            'positive_count': int,
            'negative_count': int,
            'neutral_count': int,
            'polarity_score': float (-1 to 1),
            'subjectivity': float (0 to 1)
        }
    """
    
    if not sentiment_dict or pd.isna(text):
        return {
            'positive_count': 0,
            'negative_count': 0,
            'neutral_count': 0,
            'polarity_score': 0.0,
            'subjectivity': 0.0
        }
    
    # Tokenize (simple whitespace split - text should already be cleaned)
    tokens = str(text).lower().split()
    
    positive_count = 0
    negative_count = 0
    neutral_count = 0
    
    # Check each token
    for token in tokens:
        if token in sentiment_dict:
            polarity = sentiment_dict[token]
            if polarity > 0:
                positive_count += 1
            elif polarity < 0:
                negative_count += 1
            else:
                neutral_count += 1
    
    # Calculate metrics
    total_sentiment_words = positive_count + negative_count + neutral_count
    total_words = len(tokens)
    
    # Polarity score: normalized to [-1, 1]
    if total_sentiment_words > 0:
        polarity_score = (positive_count - negative_count) / total_sentiment_words
    else:
        polarity_score = 0.0
    
    # Subjectivity: ratio of sentiment words to total words
    subjectivity = total_sentiment_words / total_words if total_words > 0 else 0.0
    
    return {
        'positive_count': positive_count,
        'negative_count': negative_count,
        'neutral_count': neutral_count,
        'polarity_score': polarity_score,
        'subjectivity': subjectivity
    }


# ============================================================
# ANALYSIS PIPELINE
# ============================================================

def analyze_sentiment_sentilex(df, text_column='clean_text', case_type='ic'):
    """
    Perform sentiment analysis using SentiLex-PT on legal texts.
    
    Args:
        df: DataFrame with legal cases
        text_column: Column containing cleaned text
        case_type: 'ic' or 'dv' (for logging)
    
    Returns:
        pd.DataFrame: Original df with added sentiment columns
    """
    
    print(f"\n{'='*80}")
    print(f"SENTILEX-PT SENTIMENT ANALYSIS - {case_type.upper()}")
    print(f"{'='*80}\n")
    
    # Load SentiLex
    # sentilex_path = Path('./resources/SentiLex-PT02/SentiLex-flex-PT02.txt')
    sentiment_dict = load_sentilex_flex()
    
    if not sentiment_dict:
        print("❌ Cannot proceed without SentiLex lexicon")
        return df
    
    # Calculate sentiment for each document
    print(f"\n📊 Calculating sentiment for {len(df)} documents...")
    
    sentiment_results = df[text_column].progress_apply(
        lambda x: calculate_sentiment_sentilex(x, sentiment_dict)
    )
    
    # Extract individual metrics into columns
    df['sentiment_positive'] = sentiment_results.apply(lambda x: x['positive_count'])
    df['sentiment_negative'] = sentiment_results.apply(lambda x: x['negative_count'])
    df['sentiment_neutral'] = sentiment_results.apply(lambda x: x['neutral_count'])
    df['sentiment_polarity'] = sentiment_results.apply(lambda x: x['polarity_score'])
    df['sentiment_subjectivity'] = sentiment_results.apply(lambda x: x['subjectivity'])
    
    # Print summary statistics
    print("\n" + "="*70)
    print("📈 SUMMARY STATISTICS")
    print("="*70)
    
    print(f"\n📊 Polarity Score:")
    print(f"   Mean: {df['sentiment_polarity'].mean():.4f}")
    print(f"   Std:  {df['sentiment_polarity'].std():.4f}")
    print(f"   Min:  {df['sentiment_polarity'].min():.4f}")
    print(f"   Max:  {df['sentiment_polarity'].max():.4f}")
    
    print(f"\n📊 Subjectivity:")
    print(f"   Mean: {df['sentiment_subjectivity'].mean():.4f}")
    print(f"   Std:  {df['sentiment_subjectivity'].std():.4f}")
    
    print(f"\n📊 Sentiment Word Counts (mean per document):")
    print(f"   Positive: {df['sentiment_positive'].mean():.2f}")
    print(f"   Negative: {df['sentiment_negative'].mean():.2f}")
    print(f"   Neutral:  {df['sentiment_neutral'].mean():.2f}")
    
    return df


# ============================================================
# STATISTICAL COMPARISON
# ============================================================

def compare_sentiment_between_cases(df_dv, df_ic, metric='sentiment_polarity'):
    """
    Compare sentiment between DV and IC cases using statistical tests.
    For all appropriate tests the confidence level is set at 95% (alpha = 0.05).
    
    Args:
        df_dv: DataFrame with domestic violence cases
        df_ic: DataFrame with contract breach cases
        metric: Sentiment metric to compare
    
    Returns:
        dict: Statistical test results
    """
    
    print(f"\n{'='*80}")
    print(f"📊 STATISTICAL COMPARISON: DV vs IC - {metric.upper()}")
    print(f"{'='*80}\n")
    
    dv_values = df_dv[metric].dropna()
    ic_values = df_ic[metric].dropna()
    
    # Descriptive statistics
    print("📈 Descriptive Statistics:")
    print(f"  DV (n={len(dv_values)}):")
    print(f"     Mean: {dv_values.mean():.4f}")
    print(f"     Std:  {dv_values.std():.4f}")
    print(f"     Median: {dv_values.median():.4f}")
    
    print(f"\n  IC (n={len(ic_values)}):")
    print(f"     Mean: {ic_values.mean():.4f}")
    print(f"     Std:  {ic_values.std():.4f}")
    print(f"     Median: {ic_values.median():.4f}")
    
    # Test for normality (Shapiro-Wilk test)
    # The Shapiro–Wilk test tests the null hypothesis that a sample x1, ..., xn came from a normally distributed population. 
    sample_size = min(5000, len(dv_values), len(ic_values)) # Shapiro-Wilk is sensitive to large sample sizes, so we limit to 5000, but I dont have that much data regardless, so it's safe to use.
    _, p_dv_normal = stats.shapiro(dv_values.sample(sample_size, random_state=42))
    _, p_ic_normal = stats.shapiro(ic_values.sample(sample_size, random_state=42))
    
    is_normal = (p_dv_normal > 0.05) and (p_ic_normal > 0.05)
    print(f"\n🔬 Normality Test (Shapiro-Wilk):")
    print(f"   DV p-value: {p_dv_normal:.6f} {'✓ Normal' if p_dv_normal > 0.05 else '✗ Non-normal'}")
    print(f"   IC p-value: {p_ic_normal:.6f} {'✓ Normal' if p_ic_normal > 0.05 else '✗ Non-normal'}")
    
    # Choose appropriate test
    if is_normal:
        # T-test (parametric)
        # The Independent Samples t Test compares the means of two independent groups in order to determine whether there is statistical evidence that the associated population means are significantly different. 
        statistic, p_value = stats.ttest_ind(dv_values, ic_values)
        test_name = "Independent t-test"
    else:
        # Mann-Whitney U test (non-parametric)
        # Similar to the t-test, the Mann-Whitney U test is used to determine whether there is a significant difference in the distribution of two independent samples.
        statistic, p_value = stats.mannwhitneyu(dv_values, ic_values, alternative='two-sided') # two-sided alternative hypothesis tests for any difference, regardless of direction (> or <).
        test_name = "Mann-Whitney U rank test"
    
    print(f"\n🧪 {test_name}:")
    print(f"   Statistic: {statistic:.4f}")
    print(f"   P-value: {p_value:.6f}")
    
    # Effect size (Cohen's d)
    # Cohen's d is a measure of effect size that indicates the standardized difference between two means.
    mean_diff = dv_values.mean() - ic_values.mean()
    pooled_std = np.sqrt(
        ((len(dv_values)-1)*dv_values.std()**2 + (len(ic_values)-1)*ic_values.std()**2) /
        (len(dv_values) + len(ic_values) - 2)
    )
    cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0
    
    print(f"\n📏 Effect Size (Cohen's d): {cohens_d:.4f}")
    effect_interpretation = (
        "negligible" if abs(cohens_d) < 0.2 else
        "small" if abs(cohens_d) < 0.5 else
        "medium" if abs(cohens_d) < 0.8 else
        "large"
    )
    print(f"   Interpretation: {effect_interpretation} effect")
    
    # Tests like t-test or Manwhitney U test test signficant difference and cohen's d quantifies the in a standardized way the magnitude of the difference.
    
    # Conclusion
    alpha = 0.05
    if p_value < alpha:
        print(f"\n✅ **Significant difference detected** (p < {alpha})")
        direction = "more positive" if mean_diff > 0 else "more negative"
        print(f"   → DV cases are **{direction}** than IC cases")
    else:
        print(f"\n❌ **No significant difference** (p ≥ {alpha})")
        print(f"   → DV and IC cases have similar sentiment")
    
    return {
        'test_name': test_name,
        'statistic': statistic,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'effect_size': effect_interpretation,
        'significant': p_value < alpha,
        'mean_dv': dv_values.mean(),
        'mean_ic': ic_values.mean()
    }


# ============================================================
# VISUALIZATION
# ============================================================

def plot_sentiment_comparison(df_dv, df_ic, metric='sentiment_polarity',
                               title="Sentiment Comparison: DV vs CB Cases"):
    """
    Create comprehensive visualization comparing sentiment between case types.
    """
    
    # Combine data
    df_dv_plot = df_dv[[metric]].copy()
    df_dv_plot['case_type'] = 'Domestic Violence'
    
    df_ic_plot = df_ic[[metric]].copy()
    df_ic_plot['case_type'] = 'Contract Breach'
    
    df_combined = pd.concat([df_dv_plot, df_ic_plot], ignore_index=True)
    
    # Create figure with 2 subplots
    fig, axes = plt.subplots(1, 2, figsize=(20, 6))
    
    # Subplot 1: Box plot
    sb.boxplot(data=df_combined, x='case_type', y=metric, ax=axes[0], palette={'Domestic Violence': 'royalblue', 'Contract Breach': 'darkorange'})
    axes[0].set_title('Distribution (Box Plot)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Case Type', fontsize=12)
    axes[0].set_ylabel(metric.replace('_', ' ').title(), fontsize=12)
    axes[0].grid(axis='y', alpha=0.3)
    
    # # Subplot 2: Violin plot
    # sb.violinplot(data=df_combined, x='case_type', y=metric, ax=axes[1], palette='Set2')
    # axes[1].set_title('Distribution (Violin Plot)', fontsize=14, fontweight='bold')
    # axes[1].set_xlabel('Case Type', fontsize=12)
    # axes[1].set_ylabel(metric.replace('_', ' ').title(), fontsize=12)
    # axes[1].grid(axis='y', alpha=0.3)
    
    # Subplot 3: Histogram
    for case_type in ['Domestic Violence', 'Contract Breach']:
        data = df_combined[df_combined['case_type'] == case_type][metric]
        axes[1].hist(data, bins=30, alpha=0.6, label=case_type, color = 'royalblue' if case_type == 'Domestic Violence' else 'darkorange')
    axes[1].set_title('Distribution (Histogram)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel(metric.replace('_', ' ').title(), fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # Save plot
    plt.savefig(
        Path(f'./eda_viz/sentiment/sentiment_comparison_{metric}.png'),
        bbox_inches='tight',
        dpi=300
    )
    
    plt.show()



In [ ]:

# # ============================================================
# # USAGE EXAMPLE
# # ============================================================

# # Enable progress bar (optional - requires tqdm)
from tqdm import tqdm
tqdm.pandas()

# # Step 1: Analyze sentiment for both case types
df_dv_sentiment = analyze_sentiment_sentilex(df_dv_bin_eda, case_type='dv')
df_ic_sentiment = analyze_sentiment_sentilex(df_ic_bin_eda, case_type='ic')

# Step 2: Compare sentiment metrics
results_polarity = compare_sentiment_between_cases(
    df_dv_sentiment,
    df_ic_sentiment,
    metric='sentiment_polarity'
)

results_subjectivity = compare_sentiment_between_cases(
    df_dv_sentiment,
    df_ic_sentiment,
    metric='sentiment_subjectivity'
)

# Step 3: Visualize
plot_sentiment_comparison(
    df_dv_sentiment,
    df_ic_sentiment,
    metric='sentiment_polarity',
    title="Sentiment Polarity: Domestic Violence vs Contract Breach (based off SentiLex-PT)"
)

plot_sentiment_comparison(
    df_dv_sentiment,
    df_ic_sentiment,
    metric='sentiment_subjectivity',
    title="Sentiment Subjectivity: Domestic Violence vs Contract Breach (based off SentiLex-PT)"
)

#### Readability

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# READABILITY CALCULATION (Portuguese-Adapted)
# Using EXISTING word/sentence/character counts
# ============================================================

def calculate_readability_from_counts(row):
    """
    Calculate readability metrics using PRE-CALCULATED counts.
    
    Expected columns in row:
        - text_length_raw: Total characters (with punctuation/spaces)
        - word_count: Total words
        - sentence_count: Total sentences
    
    Portuguese-adapted formulas (from research paper):
        ARI: 6.286 × (CH/WO) + 0.927 × (WO/SE) - 36.551
        Coleman-Liau: 5.730 × (CH/WO) - 17.1365 × (SE/WO) - 6.662
    
    Returns:
        dict: Readability metrics
        
    The Paper: Antunes, H. and Lopes, C.T. (2019) 'Analyzing the adequacy of readability indicators to a non-English language', 
    Faculdade de Engenharia da Universidade do Porto, Porto, Portugal
    INESC TEC, Porto, Portugal    
    """
    
    # Extract counts
    CH = row.get('text_length_raw', 0)  # Characters (with spaces)
    WO = row.get('word_count', 0)        # Words
    SE = row.get('sentence_count', 0)    # Sentences
    
    # Handle edge cases
    if WO == 0 or SE == 0 or CH == 0:
        return {
            'readability_ari': np.nan,
            'readability_coleman_liau': np.nan,
            'readability_avg': np.nan,
            'readability_chars_per_word': 0,
            'readability_words_per_sentence': 0
        }
    
    # Calculate ratios
    chars_per_word = CH / WO
    words_per_sentence = WO / SE
    sentences_per_word = SE / WO  # For Coleman-Liau
    
    # ============================================================
    # PORTUGUESE-ADAPTED FORMULAS
    # ============================================================
    
    # ARI (Automated Readability Index)
    ari = 6.286 * chars_per_word + 0.927 * words_per_sentence - 36.551
    
    # Coleman-Liau Index
    coleman_liau = 5.730 * chars_per_word - 17.1365 * sentences_per_word - 6.662
    
    # Average readability
    avg_readability = (ari + coleman_liau) / 2
    
    return {
        'readability_ari': ari,
        'readability_coleman_liau': coleman_liau,
        'readability_avg': avg_readability,
        'readability_chars_per_word': chars_per_word,
        'readability_words_per_sentence': words_per_sentence
    }


def interpret_readability(score):
    """
    Interpret readability score (grade level).
    
    Args:
        score: ARI or Coleman-Liau score
    
    Returns:
        str: Interpretation of difficulty level
    """
    # Based on standard ARI scale: https://readable.com/readability/automated-readability-index/
    if pd.isna(score):
        return "Unknown"
    elif score < 1:
        return "Kindergarten"
    elif score < 3:
        return "Very Easy (1st-3rd grade)"
    elif score < 5:
        return "Easy (4th-5th grade)"
    elif score < 7:
        return "Fairly Easy (6th-7th grade)"
    elif score < 9:
        return "Standard (8th-9th grade)"
    elif score < 11:
        return "Fairly Difficult (10th-11th grade)"
    elif score < 13:
        return "Difficult (12th grade- University Student)"
    else:
        return "Extremely Difficult (University Graduate+)"


# ============================================================
# ANALYSIS PIPELINE
# ============================================================

def analyze_readability(df, case_type='ic'):
    """
    Calculate readability metrics using EXISTING counts.
    
    Args:
        df: DataFrame with legal cases (must have: text_length_raw, word_count, sentence_count)
        case_type: 'ic' or 'dv' (for logging)
    
    Returns:
        pd.DataFrame: Original df with added readability columns
    """
    
    print(f"\n{'='*80}")
    print(f"READABILITY ANALYSIS (Portuguese-Adapted) - {case_type.upper()}")
    print(f"{'='*80}\n")
    
    # Verify required columns exist
    required_cols = ['text_length_raw', 'word_count', 'sentence_count']
    missing_cols = [col for col in required_cols if col not in df.columns]
    
    if missing_cols:
        print(f"❌ Missing required columns: {missing_cols}")
        print(f"   Please calculate these first!")
        return df
    
    print(f"📊 Calculating readability for {len(df)} documents...")
    print(f"   Using existing counts (text_length_raw, word_count, sentence_count)")
    
    # Calculate readability for each row
    from tqdm import tqdm
    tqdm.pandas()
    
    readability_results = df.progress_apply(
        calculate_readability_from_counts, 
        axis=1
    )
    
    # Extract metrics into separate columns
    df['readability_ari'] = readability_results.apply(lambda x: x['readability_ari'])
    df['readability_coleman_liau'] = readability_results.apply(lambda x: x['readability_coleman_liau'])
    df['readability_avg'] = readability_results.apply(lambda x: x['readability_avg'])
    df['readability_chars_per_word'] = readability_results.apply(lambda x: x['readability_chars_per_word'])
    df['readability_words_per_sentence'] = readability_results.apply(lambda x: x['readability_words_per_sentence'])
    
    # Print summary statistics
    print("\n" + "="*70)
    print("📈 SUMMARY STATISTICS")
    print("="*70)
    
    print(f"\n📊 ARI (Automated Readability Index):")
    print(f"   Mean:   {df['readability_ari'].mean():.2f}")
    print(f"   Median: {df['readability_ari'].median():.2f}")
    print(f"   Std:    {df['readability_ari'].std():.2f}")
    print(f"   Min:    {df['readability_ari'].min():.2f}")
    print(f"   Max:    {df['readability_ari'].max():.2f}")
    print(f"   Interpretation: {interpret_readability(df['readability_ari'].mean())}")
    
    print(f"\n📊 Coleman-Liau Index:")
    print(f"   Mean:   {df['readability_coleman_liau'].mean():.2f}")
    print(f"   Median: {df['readability_coleman_liau'].median():.2f}")
    print(f"   Std:    {df['readability_coleman_liau'].std():.2f}")
    print(f"   Min:    {df['readability_coleman_liau'].min():.2f}")
    print(f"   Max:    {df['readability_coleman_liau'].max():.2f}")
    print(f"   Interpretation: {interpret_readability(df['readability_coleman_liau'].mean())}")
    
    print(f"\n📊 Average Readability:")
    print(f"   Mean:   {df['readability_avg'].mean():.2f}")
    print(f"   Median: {df['readability_avg'].median():.2f}")
    print(f"   Interpretation: {interpret_readability(df['readability_avg'].mean())}")
    
    print(f"\n📊 Document Characteristics:")
    print(f"   Avg Characters per Word: {df['readability_chars_per_word'].mean():.2f}")
    print(f"   Avg Words per Sentence:  {df['readability_words_per_sentence'].mean():.2f}")
    
    return df


# ============================================================
# STATISTICAL COMPARISON (for DV vs IC comparison later)
# ============================================================

def compare_readability_between_cases(df_dv, df_ic, metric='readability_avg'):
    """
    Compare readability between DV and IC cases using statistical tests.
    
    Args:
        df_dv: DataFrame with domestic violence cases
        df_ic: DataFrame with contract breach cases
        metric: Readability metric to compare
    
    Returns:
        dict: Statistical test results
    """
    
    print(f"\n{'='*80}")
    print(f"📊 STATISTICAL COMPARISON: DV vs IC - {metric.upper()}")
    print(f"{'='*80}\n")
    
    dv_values = df_dv[metric].dropna()
    ic_values = df_ic[metric].dropna()
    
    # Descriptive statistics
    print("📈 Descriptive Statistics:")
    print(f"  DV (n={len(dv_values)}):")
    print(f"     Mean:   {dv_values.mean():.2f} ({interpret_readability(dv_values.mean())})")
    print(f"     Median: {dv_values.median():.2f}")
    print(f"     Std:    {dv_values.std():.2f}")
    
    print(f"\n  IC (n={len(ic_values)}):")
    print(f"     Mean:   {ic_values.mean():.2f} ({interpret_readability(ic_values.mean())})")
    print(f"     Median: {ic_values.median():.2f}")
    print(f"     Std:    {ic_values.std():.2f}")
    
    # Test for normality
    sample_size = min(5000, len(dv_values), len(ic_values))
    _, p_dv_normal = stats.shapiro(dv_values.sample(min(sample_size, len(dv_values)), random_state=42))
    _, p_ic_normal = stats.shapiro(ic_values.sample(min(sample_size, len(ic_values)), random_state=42))
    
    is_normal = (p_dv_normal > 0.05) and (p_ic_normal > 0.05)
    print(f"\n🔬 Normality Test (Shapiro-Wilk):")
    print(f"   DV p-value: {p_dv_normal:.6f} {'✓ Normal' if p_dv_normal > 0.05 else '✗ Non-normal'}")
    print(f"   IC p-value: {p_ic_normal:.6f} {'✓ Normal' if p_ic_normal > 0.05 else '✗ Non-normal'}")
    
    # Choose appropriate test
    if is_normal:
        statistic, p_value = stats.ttest_ind(dv_values, ic_values)
        test_name = "Independent t-test"
    else:
        statistic, p_value = stats.mannwhitneyu(dv_values, ic_values, alternative='two-sided')
        test_name = "Mann-Whitney U rank  test"
    
    print(f"\n🧪 {test_name}:")
    print(f"   Statistic: {statistic:.4f}")
    print(f"   P-value: {p_value:.6f}")
    
    # Effect size (Cohen's d)
    mean_diff = dv_values.mean() - ic_values.mean()
    pooled_std = np.sqrt(
        ((len(dv_values)-1)*dv_values.std()**2 + (len(ic_values)-1)*ic_values.std()**2) /
        (len(dv_values) + len(ic_values) - 2)
    )
    cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0
    
    print(f"\n📏 Effect Size (Cohen's d): {cohens_d:.4f}")
    effect_interpretation = (
        "negligible" if abs(cohens_d) < 0.2 else
        "small" if abs(cohens_d) < 0.5 else
        "medium" if abs(cohens_d) < 0.8 else
        "large"
    )
    print(f"   Interpretation: {effect_interpretation} effect")
    
    # Conclusion
    alpha = 0.05
    if p_value < alpha:
        print(f"\n✅ **Significant difference detected** (p < {alpha})")
        direction = "more complex" if mean_diff > 0 else "simpler"
        print(f"   → DV cases are **{direction}** than IC cases")
    else:
        print(f"\n❌ **No significant difference** (p ≥ {alpha})")
    
    return {
        'test_name': test_name,
        'statistic': statistic,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'effect_size': effect_interpretation,
        'significant': p_value < alpha,
        'mean_dv': dv_values.mean(),
        'mean_ic': ic_values.mean()
    }


# ============================================================
# VISUALIZATION
# ============================================================

def plot_readability_comparison(df_dv, df_ic, metric='readability_avg',
                                 title="Readability Comparison: DV vs IC"):
    """
    Create comprehensive visualization comparing readability.
    """
    
    # Combine data
    df_dv_plot = df_dv[[metric]].copy()
    df_dv_plot['case_type'] = 'Domestic Violence'
    
    df_ic_plot = df_ic[[metric]].copy()
    df_ic_plot['case_type'] = 'Contract Breach'
    
    df_combined = pd.concat([df_dv_plot, df_ic_plot], ignore_index=True)
    
    # Create figure with 2 subplots
    fig, axes = plt.subplots(1, 2, figsize=(20, 6))
    
    # Subplot 1: Box plot
    sb.boxplot(data=df_combined, x='case_type', y=metric, ax=axes[0], palette={'Domestic Violence': 'royalblue', 'Contract Breach': 'darkorange'})
    axes[0].set_title('Distribution (Box Plot)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Case Type', fontsize=12)
    axes[0].set_ylabel(metric.replace('_', ' ').title(), fontsize=12)
    axes[0].grid(axis='y', alpha=0.3)
    
    # # Subplot 2: Violin plot
    # sb.violinplot(data=df_combined, x='case_type', y=metric, ax=axes[1], palette='Set2')
    # axes[1].set_title('Distribution (Violin Plot)', fontsize=14, fontweight='bold')
    # axes[1].set_xlabel('Case Type', fontsize=12)
    # axes[1].set_ylabel(metric.replace('_', ' ').title(), fontsize=12)
    # axes[1].grid(axis='y', alpha=0.3)
    
    # Subplot 3: Histogram
    for case_type in ['Domestic Violence', 'Contract Breach']:
        data = df_combined[df_combined['case_type'] == case_type][metric]
        axes[1].hist(data, bins=30, alpha=0.6, label=case_type,color = 'royalblue' if case_type == 'Domestic Violence' else 'darkorange')
    axes[1].set_title('Distribution (Histogram)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel(metric.replace('_', ' ').title(), fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    plt.savefig(
        Path(f'./eda_viz/readability/readability_comparison_{metric}.png'),
        bbox_inches='tight',
        dpi=300
    )
    
    plt.show()




In [ ]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize

df_ic_bin_eda['word_count'] = df_ic_bin_eda['clean_text'].apply(lambda x: len(word_tokenize(x,language='portuguese')))
df_ic_bin_eda['sentence_count'] = df_ic_bin_eda['texto_integral_sem_decisao'].apply(lambda x: len(sent_tokenize(x,language='portuguese')))
df_ic_bin_eda['text_length_raw'] = df_ic_bin_eda['texto_integral_sem_decisao'].apply(lambda x: len(x))

df_dv_bin_eda['word_count'] = df_dv_bin_eda['clean_text'].apply(lambda x: len(word_tokenize(x,language='portuguese')))
df_dv_bin_eda['sentence_count'] = df_dv_bin_eda['texto_integral_sem_decisao'].apply(lambda x: len(sent_tokenize(x,language='portuguese')))
df_dv_bin_eda['text_length_raw'] = df_dv_bin_eda['texto_integral_sem_decisao'].apply(lambda x: len(x))

In [ ]:
# ============================================================
# USAGE EXAMPLE
# ============================================================

# # Step 1: Analyze readability (FAST - uses existing counts!)
df_ic_readability = analyze_readability(df_ic_bin_eda, case_type='ic')
df_dv_readability = analyze_readability(df_dv_bin_eda, case_type='dv')

# Step 2: Compare (when you have both DV and IC)
results_ari = compare_readability_between_cases(
    df_dv_readability,
    df_ic_readability,
    metric='readability_ari'
)

results_coleman = compare_readability_between_cases(
    df_dv_readability,
    df_ic_readability,
    metric='readability_coleman_liau'
)

results_avg = compare_readability_between_cases(
    df_dv_readability,
    df_ic_readability,
    metric='readability_avg'
)

# Step 3: Visualize
plot_readability_comparison(
    df_dv_readability,
    df_ic_readability,
    metric='readability_avg',
    title="Average Readability: DV vs IC (Portuguese-Adapted)"
)

# TODO:
 
- []  Arranjar commits e push no github (meter dados e imagens na cloud, meter so codigo no git)
- [] Fazer slides para cadeira da catia.
- [] Fazer embedding analysis com modelos portugueses sempre que possivel. Tenho tempo let's not rush.
- [] Finalizar script unico de processamento dos dados para classificação binaria e EDA